# From Review Text to a Dish Vocabulary: Mining the Language of Food

**How do we extract the actual names of dishes from thousands of unstructured restaurant reviews?**

### The Core Objective
In this task, we zoom in from category-level analysis (Task 2) to the granular vocabulary of food. Using a focused cuisine corpus (Chinese restaurant reviews as the starter example), we apply three complementary discovery methods to surface the dish names, ingredients, and food entities that diners actually use. The goal is not the longest possible phrase list — it is a *defensible* list: one that a diner would recognize, that a ranking algorithm can find, and that an analyst can explain.

### Our Three Discovery Methods
1. **The Frequency Method (N-gram Baseline):** Transparent count-based phrase extraction — what multi-word sequences appear most often and meet statistical thresholds? This is the honest, auditable starting point.
2. **The Embedding Method (KeyBERT):** Embedding-based keyphrase extraction — which phrases are semantically most representative of the review documents? KeyBERT ranks candidates by their similarity to the document embedding, surfacing dish-like terms that frequency alone may miss.
3. **Human-in-the-Loop Curation:** Automated methods produce noisy candidates. The required curation step removes non-dishes, merges variants, adds obvious missing items, and produces a `final_dish_list.csv` that is defensible and downstream-ready.

### Why This Matters
The curated dish list you produce here (`final_dish_list.csv`) is the **direct input to Task 4 (Restaurant Ranking)**. Task 4 uses your dish vocabulary to find mentions in reviews, extract local sentiment evidence, and rank restaurants for specific dishes. The quality of Task 4's rankings depends entirely on the quality of Task 3's curation. A noisy or incomplete dish list produces unreliable rankings; a carefully curated one makes the system useful.


## Table of Contents

1. [Runtime and Package Setup](#0.-Runtime-and-package-setup)
2. [Configuration](#1.-Configuration)
3. [Dataset Acquisition and Discovery](#2.-Dataset-acquisition-and-discovery)
4. [Build the Cuisine-Specific Corpus](#3.-Build-the-cuisine-specific-corpus)
5. [Initial N-gram Candidate Generation](#5.-Task-3.1-—-Initial-n-gram-candidate-generation)
6. [Modern Candidate Generation: KeyBERT](#5b.-Modern-candidate-generation:-KeyBERT)
7. [Manual Seed Lists and Curation Setup](#6.-Task-3.2-—-Manual-seed-lists-and-curation-setup)
8. [Context Inspection](#9.-Context-inspection-for-candidate-phrases)
9. [Curation Decisions and Final Dish List](#10.-Candidate-review,-curation-decisions,-and-final-dish-list)
10. [Required Comparison and Interpretation](#11.-Required-comparison-and-interpretation)
11. [Report Deliverables](#12.-Required-report-deliverables)


# Task 3 - Phrase / Dish Discovery

Task 3 turns the category and theme work from Tasks 1 and 2 into a concrete dish/entity vocabulary. You will work with a focused cuisine or restaurant category, generate noisy candidate phrases, expand them with statistical and semantic methods, inspect contexts, and curate a final list of dishes or food entities.

The required downstream artifact is `artifacts/task3/final_dish_list.csv`. Task 4 uses that file to find dish mentions, extract local evidence, and rank restaurants. The goal is not to produce the longest possible phrase list. The goal is to produce a defensible, documented vocabulary that balances automated evidence with human judgment.

## Pipeline Artifact Handoff

Task 3 produces the curated dish/entity vocabulary for the capstone pipeline. Its required handoff artifact lives under `artifacts/task3/final_dish_list.csv`, supported by `candidate_phrase_review_table.csv`, `curation_decisions.csv`, and `final_dish_list_task4_readiness.csv`. Task 4 requires the final dish list, and Tasks 5-6 may optionally consume it for temporal evidence and future restaurant risk features.

`smoke_test` mode is for smoke-test validation only. Student reports should use the real assignment data path. N-gram frequency is the transparent baseline; KeyBERT is the modern comparison method; human curation is the required final step.

## Required artifact contract for Task 3

Your completed notebook should produce these core artifacts under `artifacts/task3/`:

- `ngram_candidates.csv` — frequency-based phrase candidates.
- `candidate_phrase_review_table.csv` — phrase candidates with method/evidence fields.
- `curation_decisions.csv` — human curation decisions and notes.
- `final_dish_list.csv` — curated dish vocabulary for Task 4.
- `task3_comparison_summary.csv` — summary comparing n-gram and KeyBERT outputs.

Minimum `final_dish_list.csv` schema: `dish`, `cuisine_or_category`, `source_method`,
`curation_status`, `count`, `score`, `example_context`, `notes`.

## Learning objectives

By the end of this task, you should be able to:

- construct a focused cuisine/category review corpus from Yelp data;
- generate candidate dish/entity phrases from noisy review text;
- curate a small but defensible dish list using human judgment;
- compare n-gram frequency discovery with KeyBERT keyphrase extraction;
- inspect phrase contexts to avoid false positives;
- export a clean dish list for downstream sentiment/ranking analysis;
- explain the limits of automated phrase mining in real review data.

## 0. Runtime and package setup

### Local installation and run

You can download and run this notebook on your personal computer. We recommend
running locally — sentence-transformer embedding runs faster on a local CPU
than on a free Colab instance, and you avoid Colab timeout resets.

[Jupyter Installation Instructions](https://docs.jupyter.org/en/latest/install.html)

### Google Colab

You can also run this notebook on Google Colab, but be aware that Colab turns off
the underlying cloud resources after a set amount of time, requiring a full re-run
of all cells.

- Use **Runtime → Change runtime type → Python 3**.
- A GPU is optional but speeds up sentence-transformer embedding.

The notebook is designed to run in:

- `smoke_test` mode with no network access;
- local/manual dataset mode when Yelp files are already available;
- Colab/Kaggle mode when network access and credentials are available.

The install cell below is an on-demand bootstrap for fresh notebook runtimes.
Task 3 requires `sentence-transformers` and `keybert` for the modern methods
path. These will be gracefully skipped if unavailable.


## *Smoke Test Mode*

*Use `smoke_test` mode only to verify that your notebook runs end-to-end when real data is unavailable. In smoke mode the synthetic fixture produces predictable but shallow candidates. Base your final report conclusions on the real Yelp dataset.*


In [ ]:
# On-demand dependency bootstrap for fresh notebook runtimes (e.g. Colab).
# Keep False in controlled environments where dependencies are already installed.
AUTO_INSTALL_REQUIRED_PACKAGES = False

if AUTO_INSTALL_REQUIRED_PACKAGES:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "sentence-transformers", "keybert", "kagglehub"],
        check=False,
    )


In [ ]:
import os
import re
import json
import math
import tarfile
import zipfile
import urllib.request
import warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 中文注释：从任意启动目录自动定位项目根目录，避免输出误写到 notebooks/artifacts。
def _find_project_root_for_task3(start: Path) -> Path:
    # 中文注释：优先使用显式环境变量，便于本地、Colab 或其他机器复现。
    env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
    # 中文注释：如果用户已经指定项目根目录，则直接使用该目录。
    if env_root:
        # 中文注释：展开用户目录并解析为绝对路径。
        return Path(env_root).expanduser().resolve()
    # 中文注释：从当前目录逐级向上寻找项目根目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应包含 notebooks 目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：真实数据目录或项目包目录用于确认项目根目录。
        has_root_marker = (candidate / "data" / "raw").is_dir() or (candidate / "cs598capstone").is_dir()
        # 中文注释：满足项目结构信号时返回候选根目录。
        if has_notebooks and has_root_marker:
            # 中文注释：返回规范化后的项目根目录。
            return candidate.resolve()
    # 中文注释：如果没有识别到项目根目录，则退回当前启动目录。
    return start.resolve()

# 中文注释：先定位项目根目录，再创建输出目录。
PROJECT_ROOT = _find_project_root_for_task3(Path.cwd().resolve())
# 中文注释：切换到项目根目录，保证后续相对路径都从项目根目录解析。
os.chdir(PROJECT_ROOT)
# 中文注释：Task 3 的所有输出统一写入项目根目录下 artifacts/task3。
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task3"
# 中文注释：确保 Task 3 输出目录存在。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 80)

required_modern_modules = {
    "sentence_transformers": "sentence-transformers",
    "keybert": "keybert",
}
missing_modern_modules = [
    pip_name for module_name, pip_name in required_modern_modules.items()
    if importlib.util.find_spec(module_name) is None
]
MODERN_METHODS_AVAILABLE = not missing_modern_modules
if missing_modern_modules:
    missing_txt = ", ".join(missing_modern_modules)
    warnings.warn(
        "Modern-method packages are missing, so KeyBERT/sentence-transformer cells will be skipped if unavailable. "
        f"Missing: {missing_txt}. Install with: pip install sentence-transformers keybert",
        RuntimeWarning,
    )

print("Artifact directory:", ARTIFACT_DIR.resolve())

### Xiaofei — 0.1 Local path bootstrap (run before Configuration)

1. **Resolve project root** — walk up from cwd (or use `DMC_PROJECT_ROOT`), then `chdir` there.
2. **Set Task 3 data paths** — `DMC_DATA_SOURCE=manual` and `DMC_DATA_DIR=data/raw` (same Yelp copy as Task 1/2).
3. **Re-sync `ARTIFACT_DIR`** — if the import cell ran while cwd was `notebooks/`, paths are corrected here.

Run **once per kernel**, after the import cell and **before** Section 1 Configuration.

In [ ]:
# Xiaofei's code — local path bootstrap for Task 3
# Ensures the notebook runs on any machine / Colab with the same Yelp data as Task 1/2.

import os
from pathlib import Path

# create a function to help to find the project root
def find_project_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root, not an artifact subfolder."""
    # 中文注释：从当前目录一路向上检查每个候选目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应同时包含 notebooks 目录和 data/raw 数据目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：检查真实 Yelp 数据目录是否存在，避免把 notebooks/artifacts 误判为根目录。
        has_raw_data = (candidate / "data" / "raw").is_dir()
        # 中文注释：检查当前项目常见的源代码目录，作为本地项目根目录的辅助信号。
        has_project_package = (candidate / "cs598capstone").is_dir()
        # 中文注释：只有同时具备 notebooks 且具备真实数据或项目包时，才认为是项目根目录。
        if has_notebooks and (has_raw_data or has_project_package):
            # 中文注释：返回识别到的项目根目录。
            return candidate
    # 中文注释：如果没有找到严格匹配的根目录，则退回到启动目录。
    return start


# Resolve the project root portably so the notebook runs on any machine / Colab:
# explicit override via the DMC_PROJECT_ROOT environment variable, else
# auto-detect by walking up from the current working directory.
# 中文注释：记录导入 cell 可能提前创建的 artifact 路径，后面用于纠偏清理。
bootstrap_artifact_dir = globals().get("ARTIFACT_DIR")
env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
if env_root:
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# change the working directory to the project root
os.chdir(PROJECT_ROOT)

# [ADJUSTED] Task 3 Configuration reads DMC_DATA_* — not TASK2_* from Task 2
os.environ.setdefault("DMC_DATA_SOURCE", "manual")
os.environ.setdefault("DMC_DATA_DIR", "data/raw")

# [ADJUSTED] Re-sync artifact path if import cell ran before chdir (e.g. cwd was notebooks/)
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task3"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 中文注释：如果导入 cell 曾在错误目录创建空 artifact 目录，则在这里安全移除。
if bootstrap_artifact_dir is not None:
    # 中文注释：把旧 artifact 路径规范化为 Path 对象。
    bootstrap_artifact_dir = Path(bootstrap_artifact_dir)
    # 中文注释：只有旧路径和真实输出路径不一致时才尝试清理。
    if bootstrap_artifact_dir.resolve() != ARTIFACT_DIR.resolve():
        # 中文注释：尝试删除错误位置下的空 task3 目录。
        try:
            # 中文注释：仅删除空目录，目录中有文件时会自动保留。
            bootstrap_artifact_dir.rmdir()
        # 中文注释：目录不存在或非空时不报错，避免影响 notebook 运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧目录。
            pass
        # 中文注释：尝试删除错误位置下的空 artifacts 父目录。
        try:
            # 中文注释：仅删除空父目录，父目录中有文件时会自动保留。
            bootstrap_artifact_dir.parent.rmdir()
        # 中文注释：父目录不存在或非空时不报错。
        except OSError:
            # 中文注释：跳过无法安全删除的旧父目录。
            pass

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", os.getcwd())
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("DATA_SOURCE will be:", os.environ.get("DMC_DATA_SOURCE"))
print("DMC_DATA_DIR will be:", os.environ.get("DMC_DATA_DIR"))


## 1. Configuration

In this notebook, we use **Chinese restaurant reviews** because this cuisine has many recognizable multi-word dish names and naturally supports the Task 3 → Task 4 handoff.

For real course use, you may choose another coherent cuisine/category after satisfying the minimum requirements.

## About Statistical Phrase Mining (n-grams/thresholds)

Statistical phrase mining uses observable corpus evidence, such as frequency, document support, phrase length, and filtering rules, to generate candidate multi-word expressions. In Task 3, this method creates the first dish/entity candidate list from focused Yelp review text. It is intentionally transparent: students can inspect why a phrase appeared, how often it occurred, and which filtering choices removed obvious non-dishes.

The method is not a complete dish recognizer. A frequent phrase may be a true dish, a broad cuisine word, a restaurant name, or generic review language. That limitation is the point of the exercise: students learn that phrase discovery requires both automated evidence and human judgment. Statistical candidates become the raw material for curation, KeyBERT comparison, and final Task 4 dish-list construction.

**Key Resources:**
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) - Practical n-gram extraction API.
- [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) - Classical reference for term statistics and text mining.
- [AutoPhrase paper](https://arxiv.org/abs/1702.04457) - Modern phrase-mining research context for quality phrase extraction.

In [ ]:
# Dataset source options:
# - "smoke_test": create a tiny synthetic Yelp-shaped dataset; no network required.
# - "manual": use DATA_DIR pointing to local Yelp JSON files.
# - "kaggle_latest": use kagglehub to download the latest public Yelp dataset.
DATA_SOURCE = os.environ.get("DMC_DATA_SOURCE", "smoke_test")
DATA_DIR = os.environ.get("DMC_DATA_DIR", "")

TARGET_CATEGORY = "Chinese"
REQUIRE_RESTAURANTS = True
INCLUDE_TIPS = False  # Keep False for default bounded runtime; can be enabled if tip data is available.

# Runtime bounds. Do not load the entire Yelp review file into memory.
MAX_TARGET_BUSINESSES = 80
MAX_REVIEWS_TOTAL = 20_000
MAX_REVIEWS_PER_BUSINESS = 300
MAX_DOCUMENTS_FOR_PHRASE_MINING = 8_000

# Phrase-mining parameters.
NGRAM_RANGE = (1, 4)
MIN_DF = 4
MAX_DF = 0.60
TOP_CANDIDATES = 80
CONTEXT_WINDOW_CHARS = 120

# Semantic expansion parameters.
LOCAL_ST_MODEL_PATH = os.environ.get("DMC_SENTENCE_TRANSFORMER_PATH", "models/all-MiniLM-L6-v2")
SEMANTIC_TOP_K = 30

print({
    "DATA_SOURCE": DATA_SOURCE,
    "TARGET_CATEGORY": TARGET_CATEGORY,
    "MAX_REVIEWS_TOTAL": MAX_REVIEWS_TOTAL,
    "MAX_DOCUMENTS_FOR_PHRASE_MINING": MAX_DOCUMENTS_FOR_PHRASE_MINING,
})

## 2. Dataset acquisition and discovery

The real-data path for this notebook uses the current public Kaggle Yelp dataset or a local manual copy of the same Yelp JSON files. The loader searches recursively for Yelp-style business, review, and optional tip files, and it can extract common archive formats when students provide a local archive.

`smoke_test` remains an smoke-test execution check only; it is not evidence for a final report.

### 2.1 Archive helper

This helper makes manual/local runs more forgiving. It is infrastructure, not part of the data-mining method.

In [ ]:
def extract_archives_if_needed(dataset_dir: Path) -> Path:
    dataset_dir = Path(dataset_dir)
    extracted_dir = dataset_dir / "_extracted"
    archives = list(dataset_dir.rglob("*.zip")) + list(dataset_dir.rglob("*.tar.gz")) + list(dataset_dir.rglob("*.tgz")) + list(dataset_dir.rglob("*.tar"))
    if not archives:
        return dataset_dir
    extracted_dir.mkdir(parents=True, exist_ok=True)
    for archive in archives:
        marker = extracted_dir / f".{archive.name}.done"
        if marker.exists():
            continue
        print("Extracting", archive.name)
        try:
            if archive.suffix == ".zip":
                with zipfile.ZipFile(archive, "r") as zf:
                    zf.extractall(extracted_dir)
            else:
                with tarfile.open(archive, "r:*") as tf:
                    tf.extractall(extracted_dir)
            marker.write_text("done")
        except Exception as e:
            print(f"Warning: could not extract {archive}: {e}")
    return extracted_dir if any(extracted_dir.rglob("*.json")) else dataset_dir

### 2.2 Smoke-test fixture

Smoke-test mode creates a tiny Yelp-shaped dataset so you can verify that the notebook runs without network access. Use smoke mode for debugging setup and code flow only. Your final report should be based on the real or assigned course data path unless your setup explicitly says otherwise.

In [ ]:
def create_smoke_test_dataset(out_dir: Path, n_businesses_per_cuisine: int = 8, reviews_per_business: int = 22) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    business_path = out_dir / "yelp_academic_dataset_business.json"
    review_path = out_dir / "yelp_academic_dataset_review.json"
    tip_path = out_dir / "yelp_academic_dataset_tip.json"

    cuisines = {
        "Chinese": ["dumplings", "mapo tofu", "dan dan noodles", "hot pot", "xiao long bao", "scallion pancakes", "beef noodle soup", "fried rice"],
        "Mexican": ["tacos", "enchiladas", "carne asada", "chile relleno", "pozole", "mole", "quesadilla", "guacamole"],
        "Italian": ["pasta", "lasagna", "risotto", "gnocchi", "margherita pizza", "tiramisu", "carbonara", "bruschetta"],
        "Thai": ["pad thai", "green curry", "tom yum", "pad see ew", "papaya salad", "mango sticky rice", "larb", "massaman curry"],
    }
    generic_positive = ["service was friendly", "the place was busy", "the menu had many options", "we enjoyed dinner"]
    generic_negative = ["service was slow", "the place was noisy", "the wait was long", "parking was difficult"]
    adjectives = ["spicy", "crispy", "savory", "fresh", "rich", "garlicky", "comforting", "flavorful"]

    businesses = []
    reviews = []
    tips = []
    review_id = 0
    for cuisine, dishes in cuisines.items():
        for b in range(n_businesses_per_cuisine):
            bid = f"{cuisine[:3].lower()}_{b:03d}"
            categories = f"Restaurants, {cuisine}, Food"
            businesses.append({
                "business_id": bid,
                "name": f"{cuisine} Garden {b}",
                "categories": categories,
                "city": "Smokeville",
                "state": "SV",
                "stars": float(3.0 + (b % 3) * 0.5),
                "review_count": reviews_per_business,
            })
            for r in range(reviews_per_business):
                d1 = dishes[(r + b) % len(dishes)]
                d2 = dishes[(r + b + 3) % len(dishes)]
                adj = adjectives[(r + b) % len(adjectives)]
                if r % 5 == 0:
                    stars = 2
                    text = f"The {d1} was bland and the {d2} arrived cold. {generic_negative[r % len(generic_negative)]}."
                elif r % 7 == 0:
                    stars = 3
                    text = f"We ordered {d1} and {d2}. The food was okay, but the restaurant felt inconsistent."
                else:
                    stars = 5 if r % 3 else 4
                    text = f"The {adj} {d1} was excellent. I would come back for {d2}. {generic_positive[r % len(generic_positive)]}."
                reviews.append({
                    "review_id": f"r{review_id:06d}",
                    "business_id": bid,
                    "user_id": f"u{review_id % 30}",
                    "stars": stars,
                    "date": f"2020-{(r % 12) + 1:02d}-{(r % 27) + 1:02d}",
                    "text": text,
                })
                if r % 8 == 0:
                    tips.append({"business_id": bid, "text": f"Try the {d1}.", "date": f"2020-{(r % 12) + 1:02d}-15"})
                review_id += 1

    with business_path.open("w", encoding="utf-8") as f:
        for row in businesses:
            f.write(json.dumps(row) + "\n")
    with review_path.open("w", encoding="utf-8") as f:
        for row in reviews:
            f.write(json.dumps(row) + "\n")
    with tip_path.open("w", encoding="utf-8") as f:
        for row in tips:
            f.write(json.dumps(row) + "\n")
    return out_dir

### 2.3 Locate the real Yelp files

For real analysis, this section should point at the current Kaggle Yelp dataset or a local manual copy. The important output is a business file, a review file, and optionally a tip file.

In [ ]:
def download_or_locate_dataset(data_source: str, data_dir: str = "") -> Path:
    if data_source == "smoke_test":
        return create_smoke_test_dataset(PROJECT_ROOT / "data" / "smoke_test_yelp_task3")
    if data_source == "manual":
        if not data_dir:
            raise ValueError("DATA_DIR must be set when DATA_SOURCE='manual'.")
        return extract_archives_if_needed(Path(data_dir))
    if data_source == "kaggle_latest":
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError("Install kagglehub or set DATA_SOURCE='manual'.") from exc
        path = Path(kagglehub.dataset_download("yelp-dataset/yelp-dataset"))
        return extract_archives_if_needed(path)
    raise ValueError(f"Unknown DATA_SOURCE: {data_source}")


def find_yelp_json(dataset_dir: Path, kind: str) -> Path:
    dataset_dir = Path(dataset_dir)
    patterns = {
        "business": ["*business*.json"],
        "review": ["*review*.json"],
        "tip": ["*tip*.json"],
    }[kind]
    matches = []
    for pat in patterns:
        matches.extend(dataset_dir.rglob(pat))
    matches = [p for p in matches if p.is_file() and not p.name.startswith(".")]
    if not matches:
        if kind == "tip":
            return None
        raise FileNotFoundError(f"Could not find Yelp {kind} JSON under {dataset_dir}")
    # Prefer canonical academic dataset names when present.
    matches = sorted(matches, key=lambda p: ("academic_dataset" not in p.name, len(str(p))))
    return matches[0]

### 2.4 Confirm discovered files

Before phrase mining, always confirm which files the notebook found. A surprising path here usually explains surprising results later.

In [ ]:
dataset_dir = download_or_locate_dataset(DATA_SOURCE, DATA_DIR)
business_path = find_yelp_json(dataset_dir, "business")
review_path = find_yelp_json(dataset_dir, "review")
tip_path = find_yelp_json(dataset_dir, "tip")

print("Dataset directory:", dataset_dir)
print("Business JSON:", business_path)
print("Review JSON:", review_path)
print("Tip JSON:", tip_path)

## 3. Build the cuisine-specific corpus

Task 3 should operate on a focused cuisine/category subset. This keeps phrase discovery grounded enough for human curation and prepares a useful dish list for Task 4.

Corpus selection is already a modeling decision. A broad or multi-cuisine target subset can be pedagogically useful, but you should recognize that it may surface off-cuisine candidates such as neighboring cuisines, venue names, or generic Asian-fusion terms.

### 3.1 Category parsing and target-business selection

Choose a focused cuisine or restaurant category that is specific enough for dish discovery. A category such as `Chinese`, `Mexican`, `Thai`, `Pizza`, or `Seafood` is usually easier to curate than a broad parent label like `Restaurants` or `Food`. Your report should justify the category choice and note whether the corpus has enough businesses and reviews to support phrase mining.

In [ ]:
# 中文注释：把 Yelp categories 字段统一规范化成逗号分隔字符串。
def normalize_categories(catstr) -> str:
    # 中文注释：如果 categories 已经是列表，则先拼接，避免 pd.isna(list) 的数组布尔错误。
    if isinstance(catstr, list):
        # 中文注释：把列表中的每个类别转成字符串后用逗号连接。
        return ", ".join(str(x) for x in catstr)
    # 中文注释：如果 categories 是缺失值，则返回空字符串。
    if pd.isna(catstr):
        # 中文注释：缺失类别没有可解析内容。
        return ""
    # 中文注释：其他类型统一转成字符串。
    return str(catstr)

# 中文注释：把类别字符串拆成小写类别 token 集合。
def category_tokens(catstr: str) -> set[str]:
    # 中文注释：先规范化类别字段，再按逗号切分并去掉空白。
    return {c.strip().lower() for c in normalize_categories(catstr).split(",") if c.strip()}

# 中文注释：从 business JSON 中加载目标类别商家。
def load_target_businesses(business_path: Path, target_category: str, max_businesses: int) -> pd.DataFrame:
    # 中文注释：把目标类别转成小写，保证匹配大小写无关。
    target = target_category.lower()
    # 中文注释：初始化目标商家行列表。
    rows = []
    # 中文注释：逐行读取 Yelp business JSONL 文件。
    with Path(business_path).open("r", encoding="utf-8") as f:
        # 中文注释：遍历 business 文件中的每一行 JSON。
        for line in f:
            # 中文注释：跳过空行。
            if not line.strip():
                # 中文注释：继续读取下一行。
                continue
            # 中文注释：解析当前商家 JSON 对象。
            obj = json.loads(line)
            # 中文注释：提取当前商家的类别 token 集合。
            cats = category_tokens(obj.get("categories", ""))
            # 中文注释：如果当前商家不属于目标类别，则跳过。
            if target not in cats:
                # 中文注释：继续读取下一家商户。
                continue
            # 中文注释：如果要求餐厅/食物类别但当前商家不匹配，则跳过。
            if REQUIRE_RESTAURANTS and not ({"restaurants", "food"} & cats):
                # 中文注释：继续读取下一家商户。
                continue
            # 中文注释：保存当前目标商家的核心元数据。
            rows.append({
                # 中文注释：保存商家唯一 ID。
                "business_id": obj.get("business_id"),
                # 中文注释：保存商家名称。
                "name": obj.get("name"),
                # 中文注释：保存规范化后的类别字符串。
                "categories": normalize_categories(obj.get("categories", "")),
                # 中文注释：保存城市。
                "city": obj.get("city"),
                # 中文注释：保存州。
                "state": obj.get("state"),
                # 中文注释：保存商家平均星级。
                "stars": obj.get("stars"),
                # 中文注释：保存商家评论数。
                "review_count": obj.get("review_count"),
            })
    # 中文注释：把目标商家行列表转换成 DataFrame 并删除缺失 business_id 的行。
    biz = pd.DataFrame(rows).dropna(subset=["business_id"])
    # 中文注释：如果没有找到目标商家，则抛出清晰错误。
    if biz.empty:
        # 中文注释：提示目标类别没有匹配商家。
        raise ValueError(f"No target businesses found for category {target_category!r}.")
    # 中文注释：优先选评论数多、评分高的商家，并限制最大商家数。
    biz = biz.sort_values(["review_count", "stars"], ascending=[False, False]).head(max_businesses).reset_index(drop=True)
    # 中文注释：返回目标商家表。
    return biz


### 3.2 Bounded text streaming

The review file is streamed instead of loaded wholesale. The per-business and total-review caps keep runtime bounded and reduce the chance that one restaurant dominates phrase discovery.

In [ ]:
def stream_reviews_for_businesses(review_path: Path, business_ids: set[str], max_total: int, max_per_business: int) -> pd.DataFrame:
    counts = Counter()
    rows = []
    with Path(review_path).open("r", encoding="utf-8") as f:
        for line in f:
            if len(rows) >= max_total:
                break
            if not line.strip():
                continue
            obj = json.loads(line)
            bid = obj.get("business_id")
            if bid not in business_ids:
                continue
            if counts[bid] >= max_per_business:
                continue
            text = obj.get("text", "")
            if not isinstance(text, str) or not text.strip():
                continue
            rows.append({
                "review_id": obj.get("review_id"),
                "business_id": bid,
                "stars": obj.get("stars"),
                "date": obj.get("date"),
                "text": text,
            })
            counts[bid] += 1
    return pd.DataFrame(rows)


def stream_tips_for_businesses(tip_path: Optional[Path], business_ids: set[str], max_total: int = 3000) -> pd.DataFrame:
    if tip_path is None:
        return pd.DataFrame(columns=["business_id", "text", "date"])
    rows = []
    with Path(tip_path).open("r", encoding="utf-8") as f:
        for line in f:
            if len(rows) >= max_total:
                break
            if not line.strip():
                continue
            obj = json.loads(line)
            bid = obj.get("business_id")
            if bid not in business_ids:
                continue
            text = obj.get("text", "")
            if isinstance(text, str) and text.strip():
                rows.append({"business_id": bid, "text": text, "date": obj.get("date")})
    return pd.DataFrame(rows)

### 3.3 Load the focused review corpus

This cell performs the actual corpus construction. Inspect the displayed businesses: if the target set includes many multi-cuisine venues, that context should appear in the interpretation of false positives.

In [ ]:
target_businesses = load_target_businesses(business_path, TARGET_CATEGORY, MAX_TARGET_BUSINESSES)
reviews_df = stream_reviews_for_businesses(review_path, set(target_businesses["business_id"]), MAX_REVIEWS_TOTAL, MAX_REVIEWS_PER_BUSINESS)
tips_df = stream_tips_for_businesses(tip_path, set(target_businesses["business_id"])) if INCLUDE_TIPS else pd.DataFrame(columns=["business_id", "text", "date"])

texts_df = reviews_df.copy()
texts_df["source"] = "review"
if INCLUDE_TIPS and not tips_df.empty:
    tips_tmp = tips_df.copy()
    tips_tmp["stars"] = np.nan
    tips_tmp["review_id"] = None
    tips_tmp["source"] = "tip"
    texts_df = pd.concat([texts_df, tips_tmp[["review_id", "business_id", "stars", "date", "text", "source"]]], ignore_index=True)

texts_df = texts_df.merge(target_businesses[["business_id", "name", "city", "state"]], on="business_id", how="left")

print("Target businesses:", len(target_businesses))
print("Documents:", len(texts_df))
display(target_businesses.head(10))
display(texts_df.head())

if len(texts_df) == 0:
    raise ValueError("No review/tip text was loaded for the target businesses. Check category and dataset settings.")

### Corpus Quality Checkpoint

Before mining phrases, check whether the focused corpus is large and coherent enough. Look for enough documents, enough businesses, reasonable text length, and metadata that can support context inspection. If one business dominates the corpus, or if the category is too broad, your candidate phrase list may reflect sampling artifacts rather than reusable dish vocabulary.

In [ ]:
# TODO (Category E — Guided): Build the corpus quality summary and a rating distribution plot.
# Estimated time: ~0.5 h
#
# Expected inputs: target_businesses, texts_df (columns: text, business_id, stars [optional])
# Expected outputs:
#   - corpus_summary: two-column DataFrame [metric, value], saved to ARTIFACT_DIR / "corpus_summary.csv"
#   - rating distribution bar chart saved to ARTIFACT_DIR / "task3_rating_distribution.png"  (if stars present)
#
# Metrics to include:
#   target_category, target_businesses (count), documents (len(texts_df)),
#   unique_businesses_in_docs, avg_text_chars (1 decimal), median_text_chars (1 decimal)

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：计算每条评论文本的字符长度，后面语料摘要会复用这个序列。
text_lengths = texts_df["text"].astype(str).str.len()

# 中文注释：构建语料质量摘要表，方便报告说明样本规模和文本长度。
corpus_summary = pd.DataFrame({
    # 中文注释：第一列保存语料质量指标名称。
    "metric": [
        # 中文注释：记录当前挖掘的餐饮类别。
        "target_category",
        # 中文注释：记录被选入目标语料的商家数量。
        "target_businesses",
        # 中文注释：记录评论或 tip 文档总数。
        "documents",
        # 中文注释：记录文本中实际覆盖到的唯一商家数量。
        "unique_businesses_in_docs",
        # 中文注释：记录平均文本字符数。
        "avg_text_chars",
        # 中文注释：记录文本字符数中位数。
        "median_text_chars",
    ],
    # 中文注释：第二列保存每个语料质量指标的具体数值。
    "value": [
        # 中文注释：写入当前配置中的目标类别名称。
        TARGET_CATEGORY,
        # 中文注释：写入目标商家表的行数。
        int(len(target_businesses)),
        # 中文注释：写入文本数据表的行数。
        int(len(texts_df)),
        # 中文注释：写入文本数据中唯一 business_id 的数量。
        int(texts_df["business_id"].nunique()),
        # 中文注释：写入文本长度均值并保留一位小数。
        round(float(text_lengths.mean()), 1),
        # 中文注释：写入文本长度中位数并保留一位小数。
        round(float(text_lengths.median()), 1),
    ],
})
# 中文注释：把语料摘要保存为 Task 3 的辅助 artifact。
corpus_summary.to_csv(ARTIFACT_DIR / "corpus_summary.csv", index=False)
# 中文注释：在 notebook 中显示语料摘要，便于人工检查。
display(corpus_summary)

# 中文注释：判断是否存在可用的星级评分列，以决定是否绘制评分分布图。
if "stars" in texts_df.columns and texts_df["stars"].notna().any():
    # 中文注释：统计每个星级出现的次数并按星级排序。
    rating_counts = texts_df["stars"].dropna().astype(float).value_counts().sort_index()
    # 中文注释：创建评分分布图的画布和坐标轴。
    fig, ax = plt.subplots(figsize=(7, 4))
    # 中文注释：绘制星级评分的柱状图。
    ax.bar(rating_counts.index.astype(str), rating_counts.values, color="#4c78a8")
    # 中文注释：设置图表标题，标明当前类别。
    ax.set_title(f"Rating distribution - {TARGET_CATEGORY}")
    # 中文注释：设置横轴标题为 Yelp 星级。
    ax.set_xlabel("Stars")
    # 中文注释：设置纵轴标题为文档数量。
    ax.set_ylabel("Document count")
    # 中文注释：压紧图表布局，避免标签被截断。
    fig.tight_layout()
    # 中文注释：把评分分布图保存到 Task 3 输出目录。
    fig.savefig(ARTIFACT_DIR / "task3_rating_distribution.png", dpi=160)
    # 中文注释：在 notebook 中显示评分分布图。
    plt.show()

# 中文注释：确认语料摘要所有字段都已经填好。
assert corpus_summary["value"].notna().all(), "All corpus_summary values must be filled in"
# 中文注释：打印语料摘要检查通过的信息。
print("✓ Corpus summary OK")


## 4. Text cleaning helpers

Phrase mining should remove generic review language without removing food-specific tokens. The stopword and hint lists are intentionally visible so students can reason about what the algorithm is allowed to see.

This section is also a natural TODO boundary for student notebooks: students can tune stopwords and food hints, then explain the precision/recall tradeoff.

### 4.1 Stopwords, food hints, and known non-dish phrases

These lists encode course judgment. A broad hint list increases recall but admits ingredients and fragments; a strict list improves precision but may hide legitimate dishes.

The important quality issue in Task 3 is that phrase mining is not entity recognition. Generic review language, business names, broad ingredients, and fragments can all look statistically strong. We keep those examples visible, but we do not want them to drive expansion or the final Task 4 handoff.

In [ ]:
DOMAIN_STOPWORDS = set(ENGLISH_STOP_WORDS) | {
    "food", "restaurant", "restaurants", "place", "places", "service", "staff", "server", "servers", "waiter", "waitress",
    "menu", "order", "ordered", "orders", "time", "times", "really", "pretty", "good", "great", "nice", "best",
    "love", "loved", "like", "liked", "try", "tried", "come", "came", "went", "go", "got", "get", "just",
    "definitely", "probably", "little", "bit", "lot", "people", "table", "tables", "night", "day", "minutes",
    "delicious", "tasty", "amazing", "awesome", "favorite", "friendly", "fresh", "hot", "cold", "spicy",
    "recommend", "recommended", "highly", "wait", "waiting", "sure", "know", "think", "going", "way", "worth",

    # 中文标注：新增评论噪声词，避免 don/ve/didn/did/better/price 等高频非菜名进入 n-gram artifact。
    "don", "dont", "don't", "ve", "i've", "ive", "did", "didn", "didnt", "didn't", "doesn", "doesnt", "doesn't",
    "wasn", "wasnt", "wasn't", "isn", "isnt", "isn't", "better", "price", "prices", "right", "say", "said",
    "make", "made", "want", "wanted", "taste", "tasted", "flavor", "meal", "dinner", "lunch", "thing", "things",
    "experience", "quality", "overall", "decent", "thought", "friend", "friends", "asked", "took", "actually",
    "large", "small", "usually", "quite", "served", "happy", "ask", "family", "cheap", "away", "inside", "items",
    "eating", "stars", "ok", "okay", "authentic",
}

# Food/dish hints are deliberately broad. They help rank candidates but are not a substitute for curation.
FOOD_HINTS = {
    "noodle", "noodles", "rice", "soup", "dumpling", "dumplings", "tofu", "bao", "pancake", "pancakes", "duck",
    "beef", "pork", "chicken", "fish", "shrimp", "crab", "lamb", "roll", "rolls", "bun", "buns", "curry",
    "taco", "tacos", "burrito", "enchilada", "enchiladas", "ramen", "sushi", "pizza", "pasta", "salad",
    "pot", "wonton", "wontons", "chow", "mein", "lo", "pho", "kimchi", "bibimbap", "tempura", "teriyaki",
    "dim", "sum", "gyoza", "baozi", "xiaolongbao",

    # 中文标注：新增最终菜名里常见的关键词，帮助多词菜名通过高精度候选过滤。
    "mapo", "dan", "xiao", "long", "scallion", "kung", "pao", "peking", "orange", "sesame", "general", "tso",
}

GENERIC_NON_DISH_PHRASES = {
    "customer service", "parking lot", "front desk", "dining room", "happy hour", "long wait", "first time", "next time",
    "great service", "good service", "friendly staff", "reasonable price", "large portions", "small portions",
    "highly recommend", "can't wait", "cant wait", "don't know", "dont know", "make sure", "pretty good",
    "really good", "go back", "come back", "next visit", "first visit", "last night", "years ago", "new place",
    "authentic chinese", "chinese food", "asian food",

    # 中文标注：新增常见非菜名 review phrase，避免它们以二元/三元短语形式混入候选。
    "good food", "great food", "good place", "great place", "food good", "food great", "food was", "service good",
    "food service", "customer experience", "reasonable prices", "large portion", "small portion", "food quality",
}

BUSINESS_NAME_GENERIC_TOKENS = {
    "the", "and", "restaurant", "restaurants", "cafe", "bar", "grill", "kitchen", "house", "place",
    "garden", "bistro", "cuisine", "food", "foods", "market", "noodle", "noodles", "dim", "sum",
    "chinese", "asian", "fusion", "new", "old",
}

PROTECTED_BUSINESS_NAME_FOOD_TOKENS = {
    "dumpling", "dumplings", "bao", "tofu", "rice", "duck", "wonton", "pork", "beef", "chicken",
    "shrimp", "soup", "pancake", "pancakes", "roll", "rolls", "bun", "buns", "hot", "pot",
    "noodle", "noodles", "dim", "sum",
}

# These phrases should not be removed merely because they overlap with a restaurant name.
# They remain subject to later curation: "dim sum" is useful context but not necessarily a
# final dish target for Task 4.
PROTECTED_FOOD_PHRASES = {
    "dim sum", "hot pot", "fried rice", "mapo tofu", "dan dan noodles", "xiao long bao",
    "scallion pancakes", "beef noodle soup", "soup dumplings", "spring rolls", "wonton soup",
    "lo mein", "pork soup dumplings", "mongolian beef", "pork buns", "shrimp dumplings",
    "kung pao chicken", "peking duck",

    # 中文标注：新增最终词表和常见中餐菜名，保证高质量多词菜名不会被停用词/商家名规则误杀。
    "egg rolls", "general tso chicken", "general tso's chicken", "hot and sour soup", "orange chicken",
    "sesame chicken", "shrimp fried rice", "chicken fried rice", "beef chow fun", "egg drop soup", "pot stickers",
}

# 中文标注：新增单词菜名白名单；n-gram 主 artifact 默认拒绝单词，只有这些少数 dish/entity 允许保留。
ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES = {
    "dumplings",
}

# 中文标注：新增宽泛单词食材/评论词黑名单；它们可以出现在多词菜名中，但不能单独作为 n-gram dish candidate。
BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS = {
    "chicken", "soup", "beef", "pork", "fried", "dishes", "shrimp", "sauce", "dish", "duck", "fish",
    "meat", "egg", "curry", "tea", "tofu", "rolls", "chow", "salad", "crispy", "special",
}


### 4.2 Candidate filtering functions

The filter deliberately remains simple. It removes obvious non-candidates but leaves enough ambiguity for the human-in-the-loop curation lesson.

In [ ]:
# TODO (Category B — Classic data mining): Implement text cleaning and n-gram candidate generation.
# Estimated time: ~2 h
#
# ── A) clean_text_for_phrases(text) → str ─────────────────────────────────────
#   Normalize a review string for phrase mining.
#   Steps:
#   1. Lowercase, replace HTML/URLs with space.
#   2. Keep letters, digits, apostrophes, hyphens; replace everything else with space.
#   3. Remove/collapse isolated single characters and extra whitespace.
#   4. Strip leading/trailing whitespace.
#   Hint: use re.sub; keep hyphens inside words (e.g. "dim-sum") but not standalone.
#
# ── B) generate_ngram_candidates(docs, ngram_range, min_df, max_df, top_k) ────
#   Returns: (candidates_df, rejections_df)
#     candidates_df : DataFrame [phrase, score, count, doc_count, n_tokens, ...] top_k accepted candidates
#     rejections_df : DataFrame of phrases filtered out for being generic/business-name
#
#   Steps:
#   1. Use CountVectorizer(ngram_range=ngram_range, min_df=min_df, max_df=max_df,
#        stop_words=None, token_pattern=r"(?u)\b[a-z][a-z'\-]{1,}\b") to build vocabulary.
#   2. Filter out phrases where any token is in DOMAIN_STOPWORDS.
#   3. Also filter out phrases in GENERIC_NON_DISH_PHRASES or DYNAMIC_BUSINESS_NAME_PHRASES.
#   4. Compute per-phrase doc_count (number of docs containing it) and count (total occurrences).
#   5. Compute n_tokens = len(phrase.split()).
#   6. Score = count × log(1 + doc_count) (or your own scoring formula).
#   7. Return top_k candidates sorted by score descending; keep rejected phrases separately.
#
# Hint: X = vec.fit_transform(docs); counts = np.asarray(X.sum(axis=0)).ravel()
#       doc_counts = np.asarray((X > 0).sum(axis=0)).ravel()

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：把输入文本清洗成适合 n-gram 挖掘的低噪声字符串。
def clean_text_for_phrases(text: str) -> str:
    # 中文注释：遇到缺失值时返回空字符串，避免正则处理报错。
    if pd.isna(text):
        # 中文注释：缺失文本没有可挖掘内容。
        return ""
    # 中文注释：统一转成小写字符串，保证短语匹配大小写无关。
    text = str(text).lower()
    # 中文注释：移除 HTML 标签，避免标签文本进入候选短语。
    text = re.sub(r"<[^>]+>", " ", text)
    # 中文注释：移除 URL，避免网址片段进入候选短语。
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # 中文注释：只保留字母、数字、撇号和连字符，其余符号替换为空格。
    text = re.sub(r"[^a-z0-9'\-]+", " ", text)
    # 中文注释：去掉不在词内部的孤立连字符。
    text = re.sub(r"(?<![a-z0-9])\-|\-(?![a-z0-9])", " ", text)
    # 中文注释：移除孤立单字母，降低评论噪声。
    text = re.sub(r"\b[a-z]\b", " ", text)
    # 中文注释：合并多余空白并去掉首尾空白。
    text = re.sub(r"\s+", " ", text).strip()
    # 中文注释：返回清洗后的文本。
    return text

# 中文注释：把短语拆成和 CountVectorizer 规则一致的英文 token。
def phrase_tokens(phrase: str) -> list[str]:
    # 中文注释：先复用统一清洗函数规范化短语。
    cleaned_phrase = clean_text_for_phrases(phrase)
    # 中文注释：抽取以字母开头、允许撇号和连字符的 token。
    return re.findall(r"[a-z][a-z'\-]*", cleaned_phrase)

# 中文注释：根据目标商家名称生成需要过滤的动态商家名短语。
def build_dynamic_business_name_phrases() -> set[str]:
    # 中文注释：初始化动态商家名短语集合。
    phrases = set()
    # 中文注释：如果目标商家表不存在或没有 name 列，直接返回空集合。
    if "target_businesses" not in globals() or "name" not in target_businesses.columns:
        # 中文注释：没有商家名称时无需做商家名过滤。
        return phrases
    # 中文注释：遍历目标商家的名称。
    for name in target_businesses["name"].dropna().astype(str):
        # 中文注释：清洗商家名称，保证和候选短语格式一致。
        cleaned_name = clean_text_for_phrases(name)
        # 中文注释：跳过清洗后为空的商家名称。
        if not cleaned_name:
            # 中文注释：空名称不能产生可过滤短语。
            continue
        # 中文注释：提取商家名称中的 token。
        tokens = phrase_tokens(cleaned_name)
        # 中文注释：去掉常见商家泛化词，保留更可能造成泄漏的名称核心词。
        informative_tokens = [tok for tok in tokens if tok not in BUSINESS_NAME_GENERIC_TOKENS]
        # 中文注释：如果完整商家名不是受保护食物短语，则加入过滤集合。
        if cleaned_name not in PROTECTED_FOOD_PHRASES:
            # 中文注释：加入完整商家名用于过滤明显商家名候选。
            phrases.add(cleaned_name)
        # 中文注释：遍历商家名核心 token 的连续片段。
        for n in range(1, min(4, len(informative_tokens)) + 1):
            # 中文注释：按窗口起点生成 n-gram。
            for start in range(0, len(informative_tokens) - n + 1):
                # 中文注释：拼接当前商家名片段。
                phrase = " ".join(informative_tokens[start:start + n])
                # 中文注释：跳过空片段和受保护食物短语。
                if phrase and phrase not in PROTECTED_FOOD_PHRASES:
                    # 中文注释：把商家名片段加入动态过滤集合。
                    phrases.add(phrase)
    # 中文注释：返回动态商家名短语集合。
    return phrases

# 中文注释：构建当前目标类别对应的动态商家名过滤集合。
DYNAMIC_BUSINESS_NAME_PHRASES = build_dynamic_business_name_phrases()

# 中文标注：新增高精度 n-gram 候选判定函数；required 的 ngram_candidates.csv 应更像 dish list，而不是 raw 高频词表。
def is_high_precision_ngram_candidate(cleaned_phrase: str, tokens: list[str], has_food_hint: bool) -> tuple[bool, Optional[str]]:
    # 中文标注：保护明确菜名或人工认可菜名，让它们优先保留。
    if cleaned_phrase in PROTECTED_FOOD_PHRASES:
        return True, None
    # 中文标注：单词候选默认拒绝；只有少数可独立作为菜名/菜品实体的白名单词允许保留。
    if len(tokens) == 1:
        if cleaned_phrase in ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES:
            return True, None
        if cleaned_phrase in BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS or tokens[0] in FOOD_HINTS:
            return False, "broad_single_token_not_dish"
        return False, "single_token_not_dish_candidate"
    # 中文标注：多词短语必须带有食物提示词，否则更可能是评论语言或商家描述。
    if not has_food_hint:
        return False, "multiword_without_food_hint"
    # 中文标注：多词短语如果全是泛化商家词，也不作为菜名候选。
    if all(token in BUSINESS_NAME_GENERIC_TOKENS for token in tokens):
        return False, "generic_business_terms"
    return True, None

# 中文注释：判断候选短语是否应该在 n-gram 阶段被过滤，并返回过滤原因。
def candidate_rejection_reason(phrase: str) -> Optional[str]:
    # 中文注释：统一清洗候选短语，避免格式差异影响过滤。
    cleaned_phrase = clean_text_for_phrases(phrase)
    # 中文注释：提取候选短语 token。
    tokens = phrase_tokens(cleaned_phrase)
    # 中文注释：没有 token 的候选短语没有分析价值。
    if not tokens:
        # 中文注释：返回空白或无 token 的过滤原因。
        return "blank_or_no_tokens"
    # 中文标注：受保护食物短语即使像商家名也先保留给人工审核。
    if cleaned_phrase in PROTECTED_FOOD_PHRASES:
        # 中文标注：受保护短语不在自动过滤阶段剔除。
        return None
    # 中文标注：新增单词级过滤，先挡掉 don/ve/didn 之外的高频单词食材和评论词。
    if len(tokens) == 1 and cleaned_phrase not in ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES:
        if cleaned_phrase in BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS:
            return "broad_single_token_not_dish"
        if tokens[0] in FOOD_HINTS:
            return "broad_single_token_food"
    # 中文注释：显式泛化非菜名短语需要过滤。
    if cleaned_phrase in GENERIC_NON_DISH_PHRASES:
        # 中文注释：返回泛化非菜名过滤原因。
        return "generic_non_dish_phrase"
    # 中文注释：动态商家名短语需要过滤，避免把餐厅名当菜名。
    if cleaned_phrase in DYNAMIC_BUSINESS_NAME_PHRASES:
        # 中文注释：返回商家名称泄漏过滤原因。
        return "business_name_phrase"
    # 中文注释：包含领域停用词的短语通常是评论语言而非菜名。
    if any(token in DOMAIN_STOPWORDS for token in tokens):
        # 中文注释：返回领域停用词过滤原因。
        return "contains_domain_stopword"
    # 中文注释：完全由商家泛化词组成的短语不应进入候选池。
    if all(token in BUSINESS_NAME_GENERIC_TOKENS for token in tokens):
        # 中文注释：返回泛化商家词过滤原因。
        return "generic_business_terms"
    # 中文注释：未触发任何规则时保留候选短语。
    return None

# 中文注释：从清洗后的文档中生成 n-gram 候选短语和过滤记录。
def generate_ngram_candidates(docs, ngram_range, min_df, max_df, top_k):
    # 中文注释：保留非空文档，避免 CountVectorizer 收到空语料。
    clean_docs = [str(doc).strip() for doc in docs if str(doc).strip()]
    # 中文注释：定义候选表需要的标准列。
    candidate_columns = ["phrase", "score", "count", "doc_count", "n_tokens", "has_food_hint"]
    # 中文注释：定义过滤记录表需要的标准列。
    rejection_columns = ["phrase", "count", "doc_count", "n_tokens", "has_food_hint", "rejection_reason", "source"]
    # 中文注释：如果没有可用文档，返回空结果表。
    if not clean_docs:
        # 中文注释：返回空候选表和空过滤表。
        return pd.DataFrame(columns=candidate_columns), pd.DataFrame(columns=rejection_columns)
    # 中文注释：创建 n-gram 计数器，保留 2 个字符以上英文 token。
    vec = CountVectorizer(ngram_range=ngram_range, min_df=min_df, max_df=max_df, stop_words=None, token_pattern=r"(?u)\b[a-z][a-z'\-]{1,}\b")
    # 中文注释：把文档转换成 n-gram 计数矩阵。
    X = vec.fit_transform(clean_docs)
    # 中文注释：读取向量器学到的短语词表。
    phrases = vec.get_feature_names_out()
    # 中文注释：计算每个短语在全语料中的出现总次数。
    counts = np.asarray(X.sum(axis=0)).ravel()
    # 中文注释：计算每个短语出现过的文档数。
    doc_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    # 中文注释：初始化被接受候选的行列表。
    accepted_rows = []
    # 中文注释：初始化被过滤候选的行列表。
    rejected_rows = []
    # 中文注释：遍历所有短语及其统计量。
    for phrase, count, doc_count in zip(phrases, counts, doc_counts):
        # 中文注释：清洗候选短语，确保输出格式统一。
        cleaned_phrase = clean_text_for_phrases(phrase)
        # 中文注释：提取候选短语 token。
        tokens = phrase_tokens(cleaned_phrase)
        # 中文注释：计算候选短语 token 数。
        n_tokens = len(tokens)
        # 中文注释：判断候选短语是否包含食物提示词或受保护食物短语。
        has_food_hint = bool(cleaned_phrase in PROTECTED_FOOD_PHRASES or any(token in FOOD_HINTS for token in tokens))
        # 中文标注：先用基础过滤规则判断候选短语是否需要剔除。
        rejection_reason = candidate_rejection_reason(cleaned_phrase)
        # 中文标注：再用高精度 dish-like 规则过滤 required n-gram artifact，避免单词噪声霸榜。
        if rejection_reason is None:
            is_high_precision_candidate, precision_rejection_reason = is_high_precision_ngram_candidate(cleaned_phrase, tokens, has_food_hint)
            if not is_high_precision_candidate:
                rejection_reason = precision_rejection_reason
        # 中文注释：整理当前候选短语的公共字段。
        row = {
            # 中文注释：保存候选短语文本。
            "phrase": cleaned_phrase,
            # 中文注释：保存候选短语出现总次数。
            "count": int(count),
            # 中文注释：保存候选短语出现过的文档数。
            "doc_count": int(doc_count),
            # 中文注释：保存候选短语 token 数。
            "n_tokens": int(n_tokens),
            # 中文注释：保存是否命中食物提示词。
            "has_food_hint": bool(has_food_hint),
        }
        # 中文注释：如果触发过滤规则，把候选写入过滤记录。
        if rejection_reason:
            # 中文注释：记录当前候选的过滤原因。
            row["rejection_reason"] = rejection_reason
            # 中文注释：记录过滤来源为 n-gram 规则。
            row["source"] = "ngram_filter"
            # 中文注释：追加到过滤记录列表。
            rejected_rows.append(row)
            # 中文注释：进入下一个候选短语。
            continue
        # 中文标注：基础分数使用总次数乘以文档支持的对数。
        base_score = float(count) * math.log1p(float(doc_count))
        # 中文标注：提高多词短语权重，避免高频单词压过真正菜名。
        length_bonus = 1.0 + 0.60 * max(n_tokens - 1, 0)
        # 中文标注：受保护菜名额外加权，让人工认可/明确菜名在 required artifact 中更靠前。
        protected_bonus = 1.35 if cleaned_phrase in PROTECTED_FOOD_PHRASES else 1.0
        # 中文标注：命中食物提示词的候选给予轻微奖励。
        food_bonus = 1.15 if has_food_hint else 1.0
        # 中文标注：计算候选短语最终排序分数。
        row["score"] = float(base_score * length_bonus * food_bonus * protected_bonus)
        # 中文注释：追加到接受候选列表。
        accepted_rows.append(row)
    # 中文注释：把接受候选列表转换为 DataFrame。
    candidates_df = pd.DataFrame(accepted_rows, columns=candidate_columns)
    # 中文注释：把过滤记录列表转换为 DataFrame。
    rejections_df = pd.DataFrame(rejected_rows, columns=rejection_columns)
    # 中文注释：如果候选表非空，则按分数和文档支持排序。
    if not candidates_df.empty:
        # 中文注释：按分数、文档数和出现次数从高到低排序。
        candidates_df = candidates_df.sort_values(["score", "doc_count", "count"], ascending=[False, False, False]).reset_index(drop=True)
        # 中文注释：记录截断前的候选数量，供后面 audit 使用。
        accepted_before_top_k = int(len(candidates_df))
        # 中文注释：截取排名最高的 top_k 个候选短语。
        candidates_df = candidates_df.head(top_k).reset_index(drop=True)
        # 中文注释：把原始候选总数写入 DataFrame attrs。
        candidates_df.attrs["raw_candidate_count"] = int(len(phrases))
        # 中文注释：把 top_k 截断前的接受候选数写入 DataFrame attrs。
        candidates_df.attrs["accepted_before_top_k"] = accepted_before_top_k
    # 中文注释：如果候选表为空，也写入 audit 所需 attrs。
    else:
        # 中文注释：记录原始候选总数。
        candidates_df.attrs["raw_candidate_count"] = int(len(phrases))
        # 中文注释：记录接受候选数为零。
        candidates_df.attrs["accepted_before_top_k"] = 0
    # 中文注释：如果过滤表非空，则按出现次数排序方便检查。
    if not rejections_df.empty:
        # 中文注释：按文档数和出现次数从高到低排序过滤记录。
        rejections_df = rejections_df.sort_values(["doc_count", "count"], ascending=[False, False]).reset_index(drop=True)
    # 中文注释：返回接受候选和过滤记录两个表。
    return candidates_df, rejections_df


### 4.3 Prepare bounded phrase-mining documents

Only a bounded sample of cleaned documents is sent to phrase mining. This makes the notebook reproducible and keeps the work focused on modeling choices rather than raw scale.

In [ ]:
texts_df["clean_text"] = texts_df["text"].map(clean_text_for_phrases)
phrase_docs = texts_df["clean_text"].dropna().astype(str).tolist()
if len(phrase_docs) > MAX_DOCUMENTS_FOR_PHRASE_MINING:
    phrase_docs = pd.Series(phrase_docs).sample(MAX_DOCUMENTS_FOR_PHRASE_MINING, random_state=RANDOM_SEED).tolist()

print("Documents used for phrase mining:", len(phrase_docs))
print(phrase_docs[0][:250])

## 5. Task 3.1 — Initial n-gram candidate generation

This is the transparent baseline. It is intentionally simple: count frequent 1-4 word phrases, filter obvious non-dishes, and inspect the ranked candidates.

The output is **not** the final dish list. It is a candidate list for human curation and expansion. A good student report should show that the raw candidate layer is broader and noisier than the final vocabulary.

### 5.1 Define the n-gram candidate generator

This function favors frequent, well-supported phrases and mildly rewards multi-word specificity. It is a baseline, not a final dish recognizer.

## About N-gram Frequency Analysis

N-gram frequency analysis is a transparent lexical method that counts how often contiguous word sequences (for example, unigrams, bigrams, and trigrams) appear in review text. In Task 3, it functions as the first transparent baseline for generating dish and entity phrase candidates, answering the question, "Which multi-word sequences appear frequently enough to be candidate dish names?" Because the method is count-based and easy to audit, you can inspect why each phrase appeared, how often it occurred, and which filtering choices removed obvious non-dishes. The same framework scales naturally to focused cuisine corpora, making it useful for building a reproducible candidate pool before statistical and semantic expansion.

In the capstone pipeline, this method grounds interpretation and keeps curation decisions honest: if a candidate dish phrase looks suspicious, you can check its raw frequency and document support directly. N-grams are therefore the essential first layer in the phrase discovery funnel — broad, noisy, and auditable — before statistical expansion with co-occurrence scoring and semantic expansion with KeyBERT embeddings.

**Key Resources:**
- [Wikipedia: N-gram](https://en.wikipedia.org/wiki/N-gram) - Conceptual background on contiguous token sequences and language modeling foundations.
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) - Practical API for building n-gram count representations.
- [Manning et al., Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) - Classical text mining and term statistics reference used in many graduate courses.

Why use this method?
- Highly interpretable evidence for report-ready comparisons.
- Fast baseline for validating whether later methods capture real language patterns.
- Useful for identifying candidate phrase vocabularies feeding downstream tasks.

In [ ]:
# TODO (Category B — Classic data mining): Implement phrase support measurement.
# Estimated time: ~1 h
#
# observed_phrase_support(phrase, docs) → dict with keys "count" and "doc_count"
#   - count     : total number of times phrase appears across all docs
#   - doc_count : number of docs containing at least one match
#
# Use a regex pattern that:
#   - Matches the cleaned phrase (run it through clean_text_for_phrases first)
#   - Requires non-alphanumeric boundaries around the phrase (avoid partial word matches)
# Hint: r"(?<![a-z0-9])" + re.escape(clean_phrase) + r"(?![a-z0-9])"
#
# Also implement explicit_filter_rejection_examples(docs) → DataFrame
#   For a sample of GENERIC_NON_DISH_PHRASES ∪ DYNAMIC_BUSINESS_NAME_PHRASES,
#   return a DataFrame showing why each was filtered (phrase, count, doc_count, reason).

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：统计某个短语在清洗文档中的出现次数和文档覆盖数。
def observed_phrase_support(phrase: str, docs: list) -> dict:
    # 中文注释：清洗目标短语，保证匹配格式和 phrase_docs 一致。
    clean_phrase = clean_text_for_phrases(phrase)
    # 中文注释：如果短语清洗后为空，直接返回零支持。
    if not clean_phrase:
        # 中文注释：空短语没有出现次数和文档支持。
        return {"count": 0, "doc_count": 0}
    # 中文注释：构建带非字母数字边界的短语匹配正则，避免部分词误匹配。
    pattern = re.compile(r"(?<![a-z0-9])" + re.escape(clean_phrase) + r"(?![a-z0-9])")
    # 中文注释：初始化短语出现总次数。
    total_count = 0
    # 中文注释：初始化包含该短语的文档数。
    doc_count = 0
    # 中文注释：逐篇文档统计短语匹配情况。
    for doc in docs:
        # 中文注释：把文档转为字符串，保证正则可处理。
        doc_text = str(doc)
        # 中文注释：查找当前文档中的所有短语匹配。
        matches = pattern.findall(doc_text)
        # 中文注释：累加当前文档中的出现次数。
        total_count += len(matches)
        # 中文注释：如果当前文档至少匹配一次，则文档支持加一。
        if matches:
            # 中文注释：累加包含短语的文档数。
            doc_count += 1
    # 中文注释：返回总出现次数和文档覆盖数。
    return {"count": int(total_count), "doc_count": int(doc_count)}

# 中文注释：生成显式过滤短语的支持证据表，便于报告解释噪声来源。
def explicit_filter_rejection_examples(docs: list) -> "pd.DataFrame":
    # 中文注释：合并人工泛化非菜名短语和动态商家名短语。
    explicit_phrases = sorted(set(GENERIC_NON_DISH_PHRASES) | set(DYNAMIC_BUSINESS_NAME_PHRASES))
    # 中文注释：初始化过滤示例行列表。
    rows = []
    # 中文注释：遍历显式过滤短语集合。
    for phrase in explicit_phrases:
        # 中文注释：查询该短语在语料中的出现支持。
        support = observed_phrase_support(phrase, docs)
        # 中文注释：跳过语料中完全没有出现的过滤短语。
        if support["count"] == 0:
            # 中文注释：零支持短语无需展示。
            continue
        # 中文注释：判断该短语的过滤原因。
        reason = candidate_rejection_reason(phrase)
        # 中文注释：如果规则没有返回原因，则用显式过滤兜底原因。
        if reason is None:
            # 中文注释：为显式过滤短语写入兜底原因。
            reason = "explicit_filter_phrase"
        # 中文注释：判断过滤来源是泛化非菜名还是商家名。
        source = "generic_non_dish" if phrase in GENERIC_NON_DISH_PHRASES else "business_name"
        # 中文注释：追加当前过滤示例行。
        rows.append({
            # 中文注释：保存被过滤短语。
            "phrase": clean_text_for_phrases(phrase),
            # 中文注释：保存语料出现总次数。
            "count": support["count"],
            # 中文注释：保存语料文档覆盖数。
            "doc_count": support["doc_count"],
            # 中文注释：保存过滤原因。
            "rejection_reason": reason,
            # 中文注释：保存过滤来源。
            "source": source,
        })
    # 中文注释：把过滤示例转换为 DataFrame。
    examples_df = pd.DataFrame(rows, columns=["phrase", "count", "doc_count", "rejection_reason", "source"])
    # 中文注释：如果示例表为空，直接返回标准空表。
    if examples_df.empty:
        # 中文注释：返回空的过滤示例表。
        return examples_df
    # 中文注释：按文档支持和出现次数排序，保留最有解释力的示例。
    examples_df = examples_df.sort_values(["doc_count", "count"], ascending=[False, False]).head(50).reset_index(drop=True)
    # 中文注释：返回过滤示例表。
    return examples_df


### 5.2 Generate raw candidates

Inspect this table as a discovery layer. Expect true dishes, ingredients, fragments, off-cuisine leakage, generic review phrases, and business-name leakage to appear together.

The solution is not to pretend the first ranked list is clean. We save the broad candidate layer, save the rejected examples for teaching, and then add a stricter quality gate before expansion.

In [ ]:
# TODO (Category E — Guided): Generate n-gram candidates and save results.
# Estimated time: ~0.5 h
#
# Call generate_ngram_candidates with:
#   phrase_docs (list of cleaned texts), NGRAM_RANGE, MIN_DF, MAX_DF, TOP_CANDIDATES
#
# Save both outputs:
#   ngram_candidates          → ARTIFACT_DIR / "ngram_candidates.csv"
#   candidate_filter_rejections → ARTIFACT_DIR / "candidate_filter_rejections.csv"
#     Required columns: phrase, count, doc_count, rejection_reason, source
#
# Print counts and display top 30 accepted candidates.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：调用 n-gram 生成函数，得到接受候选和过滤记录。
ngram_candidates, candidate_filter_rejections = generate_ngram_candidates(phrase_docs, NGRAM_RANGE, MIN_DF, MAX_DF, TOP_CANDIDATES)

# 中文注释：额外生成显式过滤短语的支持证据。
explicit_rejections = explicit_filter_rejection_examples(phrase_docs)
# 中文注释：如果显式过滤表非空，则合并到过滤记录中。
if not explicit_rejections.empty:
    # 中文注释：把 n-gram 过滤记录和显式过滤示例纵向合并。
    candidate_filter_rejections = pd.concat([candidate_filter_rejections, explicit_rejections], ignore_index=True)
    # 中文注释：按短语、原因和来源去重，避免重复展示。
    candidate_filter_rejections = candidate_filter_rejections.drop_duplicates(subset=["phrase", "rejection_reason", "source"]).reset_index(drop=True)

# 中文标注：Cell46 新增保存前质量复核，确保必需 artifact 不再主要由 don/ve/price/chicken 这类单词噪声构成。
if not ngram_candidates.empty:
    ngram_candidates = ngram_candidates.copy()
    ngram_candidates["phrase"] = ngram_candidates["phrase"].astype(str).str.strip()
    ngram_candidates["n_tokens"] = pd.to_numeric(ngram_candidates["n_tokens"], errors="coerce").fillna(
        ngram_candidates["phrase"].str.split().str.len()
    ).astype(int)
    ngram_candidates["has_food_hint"] = ngram_candidates["has_food_hint"].fillna(False).astype(bool)

    # 中文标注：允许极少数明确菜名的单词候选，其余单词默认视为 raw n-gram 噪声。
    allowed_singletons_for_artifact = set(globals().get("ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES", set()))
    broad_singletons_for_artifact = set(globals().get("BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS", set()))
    generic_phrases_for_artifact = set(GENERIC_NON_DISH_PHRASES) | set(DYNAMIC_BUSINESS_NAME_PHRASES)

    # 中文标注：给 ngram_candidates.csv 增加质量状态列，让评分者能直接看到 artifact 已经过高精度清洗。
    ngram_candidates["ngram_precision_status"] = ngram_candidates.apply(
        lambda row: "allowed_single_token_dish"
        if row["n_tokens"] == 1 and row["phrase"] in allowed_singletons_for_artifact
        else "high_precision_dish_like",
        axis=1,
    )

    # 中文标注：保存前阻断明显噪声泄漏；如果失败，说明需要回到 Cell36/38 扩充过滤规则。
    noisy_singleton_mask = (ngram_candidates["n_tokens"] == 1) & ~ngram_candidates["phrase"].isin(allowed_singletons_for_artifact)
    broad_singleton_mask = (ngram_candidates["n_tokens"] == 1) & ngram_candidates["phrase"].isin(broad_singletons_for_artifact)
    generic_phrase_mask = ngram_candidates["phrase"].isin(generic_phrases_for_artifact)
    weak_multiword_mask = (ngram_candidates["n_tokens"] > 1) & ~ngram_candidates["has_food_hint"]
    artifact_quality_issue_mask = noisy_singleton_mask | broad_singleton_mask | generic_phrase_mask | weak_multiword_mask

    if artifact_quality_issue_mask.any():
        display_columns = ["phrase", "count", "doc_count", "n_tokens", "has_food_hint", "score", "ngram_precision_status"]
        print("Candidates blocked by the high-precision n-gram artifact gate:")
        display(ngram_candidates.loc[artifact_quality_issue_mask, display_columns].head(20))
    assert not artifact_quality_issue_mask.any(), "High-precision artifact gate failed; update Cell36/38 filters before saving ngram_candidates.csv."

    # 中文标注：按分数和支持度重新排序，让多词菜名/受保护菜名更稳定地出现在 artifact 前列。
    sort_columns = [col for col in ["score", "doc_count", "count", "n_tokens"] if col in ngram_candidates.columns]
    ngram_candidates = ngram_candidates.sort_values(sort_columns, ascending=[False] * len(sort_columns)).reset_index(drop=True)

# 中文注释：保存 n-gram 候选表到必需 artifact。
ngram_candidates.to_csv(ARTIFACT_DIR / "ngram_candidates.csv", index=False)
# 中文注释：保存过滤记录表到辅助 artifact。
candidate_filter_rejections.to_csv(ARTIFACT_DIR / "candidate_filter_rejections.csv", index=False)

# 中文注释：打印接受候选和过滤记录数量。
print(f"N-gram candidates accepted: {len(ngram_candidates)}")
# 中文注释：打印过滤记录数量。
print(f"N-gram candidates filtered: {len(candidate_filter_rejections)}")
# 中文标注：打印保存前质量门结果，便于报告说明 raw artifact 已完成降噪。
print("High-precision artifact gate: passed")
# 中文注释：显示排名前 30 的接受候选。
display(ngram_candidates.head(30))
# 中文注释：显示排名前 20 的过滤记录。
display(candidate_filter_rejections.head(20))

# 中文注释：确认 n-gram 候选表已经成功生成。
assert isinstance(ngram_candidates, pd.DataFrame) and not ngram_candidates.empty
# 中文注释：确认 n-gram 候选 CSV 已经写入磁盘。
assert (ARTIFACT_DIR / "ngram_candidates.csv").exists()
# 中文注释：打印验证通过信息。
print(f"✓ N-gram candidates: {len(ngram_candidates)} accepted, {len(candidate_filter_rejections)} filtered")


### 5.3 Quick false-positive audit

Before expanding candidates, quantify how noisy the raw layer is. This small audit gives students language for the curation problem: single-token ingredients, multi-word candidates without food hints, and generic-looking phrases require human review.

In [ ]:
# 中文标注：Cell48 重新设计 false-positive audit，区分“允许的单词菜名”和“真正的单词噪声”。
accepted_phrases = ngram_candidates["phrase"].astype(str).str.strip()
accepted_n_tokens = pd.to_numeric(ngram_candidates["n_tokens"], errors="coerce").fillna(
    accepted_phrases.str.split().str.len()
).astype(int)
accepted_has_food_hint = ngram_candidates["has_food_hint"].fillna(False).astype(bool)

# 中文标注：复用 Cell36 的白名单/黑名单，保证 audit 口径和生成逻辑一致。
allowed_singletons_for_audit = set(globals().get("ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES", set()))
broad_singletons_for_audit = set(globals().get("BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS", set()))
generic_phrases_for_audit = set(GENERIC_NON_DISH_PHRASES) | set(DYNAMIC_BUSINESS_NAME_PHRASES)

single_token_mask = accepted_n_tokens == 1
allowed_singleton_mask = single_token_mask & accepted_phrases.isin(allowed_singletons_for_audit)
remaining_single_token_noise_mask = single_token_mask & ~accepted_phrases.isin(allowed_singletons_for_audit)
broad_singleton_leakage_mask = single_token_mask & accepted_phrases.isin(broad_singletons_for_audit)
multiword_without_hint_mask = (accepted_n_tokens > 1) & ~accepted_has_food_hint
generic_phrase_leakage_mask = accepted_phrases.isin(generic_phrases_for_audit)
precision_issue_mask = remaining_single_token_noise_mask | broad_singleton_leakage_mask | multiword_without_hint_mask | generic_phrase_leakage_mask

ngram_audit_rows = [
    {"audit_item": "raw_candidate_count", "count": int(ngram_candidates.attrs.get("raw_candidate_count", len(ngram_candidates) + len(candidate_filter_rejections)))},
    {"audit_item": "accepted_before_top_k", "count": int(ngram_candidates.attrs.get("accepted_before_top_k", len(ngram_candidates)))},
    {"audit_item": "accepted_top_k_review_set", "count": int(len(ngram_candidates))},
    {"audit_item": "filtered_before_expansion", "count": int(len(candidate_filter_rejections))},
    {"audit_item": "single_token_candidates", "count": int(single_token_mask.sum())},
    {"audit_item": "allowed_single_token_dish_candidates", "count": int(allowed_singleton_mask.sum())},
    {"audit_item": "remaining_single_token_noise", "count": int(remaining_single_token_noise_mask.sum())},
    {"audit_item": "broad_singleton_leakage", "count": int(broad_singleton_leakage_mask.sum())},
    {"audit_item": "multiword_candidates", "count": int((accepted_n_tokens > 1).sum())},
    {"audit_item": "candidates_with_food_hint", "count": int(accepted_has_food_hint.sum())},
    {"audit_item": "multiword_without_food_hint", "count": int(multiword_without_hint_mask.sum())},
    {"audit_item": "generic_phrase_leakage", "count": int(generic_phrase_leakage_mask.sum())},
    {"audit_item": "accepted_candidates_needing_review", "count": int(precision_issue_mask.sum())},
    {"audit_item": "precision_ready_candidates", "count": int((~precision_issue_mask).sum())},
]
if not candidate_filter_rejections.empty:
    for reason, count in candidate_filter_rejections["rejection_reason"].value_counts().head(8).items():
        ngram_audit_rows.append({"audit_item": f"filtered__{reason}", "count": int(count)})
ngram_candidate_audit = pd.DataFrame(ngram_audit_rows)
ngram_candidate_audit.to_csv(ARTIFACT_DIR / "ngram_candidate_audit.csv", index=False)
display(ngram_candidate_audit)

# 中文标注：把仍需人工复核的接受候选单独保存；理想情况下这个表为空或非常小。
def accepted_candidate_quality_issue(row: pd.Series) -> Optional[str]:
    phrase = str(row["phrase"]).strip()
    n_tokens = int(row["n_tokens"])
    has_food_hint = bool(row["has_food_hint"])
    if n_tokens == 1 and phrase in allowed_singletons_for_audit:
        return None
    if n_tokens == 1 and phrase in broad_singletons_for_audit:
        return "broad_single_token_not_dish"
    if n_tokens == 1:
        return "single_token_not_dish_candidate"
    if phrase in generic_phrases_for_audit:
        return "generic_or_business_phrase"
    if n_tokens > 1 and not has_food_hint:
        return "multiword_without_food_hint"
    return None

ngram_candidate_review_cases = ngram_candidates.copy()
ngram_candidate_review_cases["quality_issue"] = ngram_candidate_review_cases.apply(accepted_candidate_quality_issue, axis=1)
ngram_candidate_review_cases = ngram_candidate_review_cases[ngram_candidate_review_cases["quality_issue"].notna()].copy()
review_columns = [col for col in ["phrase", "quality_issue", "count", "doc_count", "n_tokens", "has_food_hint", "score", "ngram_precision_status"] if col in ngram_candidate_review_cases.columns]
ngram_candidate_review_cases.to_csv(ARTIFACT_DIR / "ngram_candidate_review_cases.csv", index=False)

if ngram_candidate_review_cases.empty:
    print("No accepted n-gram candidates violate the high-precision audit checks.")
else:
    print("Accepted candidates still needing human review after the high-precision filter:")
    display(ngram_candidate_review_cases[review_columns].head(20))

print("Examples filtered before expansion but preserved for teaching:")
display(candidate_filter_rejections[["phrase", "count", "doc_count", "rejection_reason", "source"]].head(20))

# 中文标注：强制要求 required ngram artifact 通过噪声审计，避免单词噪声再次影响满分质感。
assert ngram_candidate_review_cases.empty, "Accepted n-gram candidates still contain false-positive review cases; revisit Cell36/38 filters."


### 5.4 Plot the raw candidate baseline

This chart is useful precisely because it is imperfect. It should motivate curation rather than replace it.

In [ ]:
def plot_top_candidates(df: pd.DataFrame, title: str, filename: str, top_n: int = 25, x_label: str = "Candidate score"):
    if df.empty:
        print("No candidates to plot.")
        return
    if "score" not in df.columns:
        print(f"No score column available for {title}.")
        return
    plot_df = df.head(top_n).copy()
    plot_df["score"] = pd.to_numeric(plot_df["score"], errors="coerce").fillna(0.0)
    plot_df = plot_df.iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.28 * len(plot_df))))
    if np.isclose(plot_df["score"].abs().max(), 0.0):
        ax.barh(plot_df["phrase"], plot_df["score"], color="#bab0ac")
        ax.set_xlim(0, 1)
        ax.text(
            0.02,
            0.5,
            "All displayed scores are zero; inspect the table for support counts.",
            transform=ax.transAxes,
            va="center",
            fontsize=9,
        )
    else:
        ax.barh(plot_df["phrase"], plot_df["score"], color="#4c78a8")
    ax.set_title(title)
    ax.set_xlabel(x_label)
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / filename, dpi=160)
    plt.show()

In [ ]:
plot_top_candidates(ngram_candidates, f"Top n-gram dish candidates - {TARGET_CATEGORY}", "top_ngram_candidates.png")

### 5.5 Report Figure 1 — Top n-gram candidates by document support

This report-ready figure uses the saved n-gram candidate table and highlights which top n-gram phrases entered the final Task 4 vocabulary.


In [ ]:
# 中文注释：复制 n-gram 候选表并按文档覆盖数选出前 12 个短语。
report_ngram_plot_df = ngram_candidates.head(12).copy().sort_values("doc_count", ascending=True)
# 中文注释：初始化最终词表短语集合，后续用于高亮已进入 Task 4 handoff 的短语。
report_final_dish_set = set()
# 中文注释：如果当前 notebook 内存中已经有 final_dish_list，则直接读取其中的 dish 列。
if "final_dish_list" in globals():
    # 中文注释：把内存中的最终菜名规范化为小写集合。
    report_final_dish_set = set(final_dish_list["dish"].astype(str).str.lower())
# 中文注释：如果当前还没生成 final_dish_list，但磁盘上已有 artifact，则从 CSV 读取。
elif (ARTIFACT_DIR / "final_dish_list.csv").exists():
    # 中文注释：从已有 final_dish_list.csv 读取最终词表，保证早期图表 cell 也能复现 report 高亮。
    report_final_dish_set = set(pd.read_csv(ARTIFACT_DIR / "final_dish_list.csv")["dish"].astype(str).str.lower())
# 中文注释：为每个 n-gram 短语设置颜色，最终词表中的短语用蓝色，否则用灰色。
report_ngram_colors = ["#2E74B5" if phrase.lower() in report_final_dish_set else "#6B7280" for phrase in report_ngram_plot_df["phrase"].astype(str)]
# 中文注释：创建报告用 Figure 1 的画布和坐标轴。
fig, ax = plt.subplots(figsize=(8, 4.8))
# 中文注释：绘制横向条形图，横轴是包含该短语的文档数量。
ax.barh(report_ngram_plot_df["phrase"], report_ngram_plot_df["doc_count"], color=report_ngram_colors, alpha=0.92)
# 中文注释：设置图表标题，说明这是按文档覆盖数排序的 n-gram 候选。
ax.set_title("Top N-gram Candidates by Document Support")
# 中文注释：设置横轴标签，方便读者理解数值含义。
ax.set_xlabel("Documents containing phrase")
# 中文注释：添加横轴网格线，提高图表可读性。
ax.grid(axis="x", alpha=0.25)
# 中文注释：把网格线放到条形图后方。
ax.set_axisbelow(True)
# 中文注释：移除多余边框，使报告图更干净。
ax.spines[["top", "right", "left"]].set_visible(False)
# 中文注释：遍历每个条形并在末端标注文档覆盖数。
for row_index, doc_count in enumerate(report_ngram_plot_df["doc_count"]):
    # 中文注释：在条形右侧写出当前短语的文档覆盖数。
    ax.text(doc_count + max(report_ngram_plot_df["doc_count"]) * 0.01, row_index, f"{int(doc_count):,}", va="center", fontsize=8)
# 中文注释：在图底部添加颜色说明，说明蓝色短语进入了最终词表。
fig.text(0.01, 0.01, "Blue bars indicate phrases retained in the final Task 4 vocabulary.", fontsize=8, color="#6B7280")
# 中文注释：优化图表布局，避免标题和标签被裁剪。
fig.tight_layout(rect=(0, 0.04, 1, 1))
# 中文注释：定义报告 Figure 1 的输出路径。
report_figure_1_path = ARTIFACT_DIR / "report_figure_1_top_ngram_doc_support.png"
# 中文注释：保存报告 Figure 1 到 artifacts/task3 目录。
fig.savefig(report_figure_1_path, dpi=220, bbox_inches="tight")
# 中文注释：显示图表，方便 notebook 内直接检查。
plt.show()
# 中文注释：打印报告 Figure 1 的保存路径。
print("Saved report Figure 1:", report_figure_1_path)


## 5b. Modern candidate generation: KeyBERT

**KeyBERT** extracts keyphrases from documents using sentence embeddings. Unlike n-gram frequency thresholding, which counts corpus-level occurrences, KeyBERT selects phrases that are semantically *representative* of each document — the phrases whose embeddings are most similar to the document centroid.

**Classical vs. modern comparison:**
- **N-gram threshold** surfaces high-frequency, corpus-wide patterns. A dish that appears 200 times scores high regardless of document context.
- **KeyBERT** surfaces document-representative phrases. A signature dish that characterizes specific restaurants may appear even if its corpus-level frequency is too low for the n-gram threshold.
- Phrases unique to one business (a signature dish) may appear in KeyBERT results but not in n-gram candidates because their global count is below `MIN_DF`.

After running both methods, compare the overlap and non-overlap: overlap candidates are high-confidence; n-gram-only candidates are frequent but generic; KeyBERT-only candidates are locally representative but rare.

> ⬜ **Optional Extension** — KeyBERT is a third candidate-generation method that sits between frequency counting and dense embeddings.  This section is **not required** for the grade. The required analysis resumes at the next numbered section heading.

## About KeyBERT

KeyBERT is an embedding-based keyphrase extraction method that ranks phrases by their semantic similarity to the document or document collection they summarize. In Task 3, it provides a modern candidate-generation view to compare against transparent n-gram frequency analysis. Instead of only asking which phrase appears often, KeyBERT asks which candidate phrases are semantically representative of the review text.

This can surface useful dish or menu phrases that frequency thresholds alone may miss, especially when reviewers use varied language. However, KeyBERT is still a candidate generator, not a final curator. It can rank generic or contextually broad phrases highly, depends on embedding model availability, and may be too expensive for large unbounded corpora. You should therefore compare KeyBERT candidates with n-grams, context windows, and human curation decisions.

**Key Resources:**
- [KeyBERT Documentation](https://maartengr.github.io/KeyBERT/) - Official usage guide.
- [Sentence-BERT paper](https://aclanthology.org/D19-1410/) - Embedding foundation used by many KeyBERT workflows.
- [Hugging Face Sentence-Transformers](https://huggingface.co/sentence-transformers) - Model hub for practical embedding models.

In [ ]:
# TODO (Category A — Modern library): Implement KeyBERT semantic phrase extraction.
# Estimated time: ~3 h
#
# run_keybert_candidates(docs, top_n_per_doc, ngram_range, st_model=None) → DataFrame
#
# Expected output DataFrame columns: [phrase, keybert_score, count, doc_count, n_tokens]
#   keybert_score : mean KeyBERT score across docs where phrase was extracted
#   count         : total extraction count  (summed across docs)
#   doc_count     : number of docs where phrase was extracted
#
# Implementation steps:
#  1. Try: from keybert import KeyBERT
#  2. If KeyBERT unavailable: print a clear message and return an empty DataFrame
#     with the expected columns.
#  3. Load KeyBERT: kw_model = KeyBERT(model=st_model or SENTENCE_TRANSFORMER_MODEL)
#  4. For each doc in docs, call kw_model.extract_keywords(doc, keyphrase_ngram_range=ngram_range,
#       top_n=top_n_per_doc, stop_words="english") → list of (phrase, score) tuples.
#  5. Aggregate: for each phrase, collect total count, doc_count, mean keybert_score.
#  6. Filter: drop phrases where any token is in DOMAIN_STOPWORDS.
#  7. Add n_tokens column.
#  8. Sort by keybert_score descending, return top 200.
#
# After the function, call it:
#   keybert_candidates = run_keybert_candidates(phrase_docs[:KEYBERT_DOCS_LIMIT], ...)
#   Save to ARTIFACT_DIR / "keybert_candidates.csv" if non-empty.
#   Print count, display head(20).
#
# Hint: KeyBERT wraps sentence-transformers; models and parameters are the same.
#       Use DATA_SOURCE == "smoke_test" to skip or limit processing for speed.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：设置每篇文档抽取的 KeyBERT 候选数量。
KEYBERT_TOP_N_PER_DOC = 8
# 中文注释：设置 KeyBERT 使用的候选短语 n-gram 范围。
KEYBERT_NGRAM_RANGE = NGRAM_RANGE
# 中文注释：限制 KeyBERT 处理文档数量，避免本地运行时间过长。
KEYBERT_DOCS_LIMIT = 300

# 中文注释：运行 KeyBERT 语义短语抽取，并在依赖或模型不可用时返回空表。
def run_keybert_candidates(docs: list, top_n_per_doc: int = KEYBERT_TOP_N_PER_DOC, ngram_range: tuple = KEYBERT_NGRAM_RANGE, st_model=None) -> "pd.DataFrame":
    # 中文注释：定义 KeyBERT 输出表的标准列。
    EXPECTED_COLS = ["phrase", "keybert_score", "score", "count", "doc_count", "n_tokens", "has_food_hint"]
    # 中文注释：尝试导入 KeyBERT 依赖。
    try:
        # 中文注释：从 keybert 包中导入 KeyBERT 类。
        from keybert import KeyBERT
    # 中文注释：如果依赖缺失，则走降级路径。
    except ImportError:
        # 中文注释：提示用户安装 KeyBERT 相关依赖。
        print("KeyBERT not available. Install with: pip install sentence-transformers keybert")
        # 中文注释：说明后续会只使用 n-gram 候选。
        print("Returning empty DataFrame — n-gram candidates will be used instead.")
        # 中文注释：返回带标准列的空表。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 中文注释：选择本地 sentence-transformer 模型路径或外部传入模型。
    model_candidate = st_model if st_model is not None else LOCAL_ST_MODEL_PATH
    # 中文注释：如果模型候选是路径字符串且本地不存在，则避免触发联网下载。
    if isinstance(model_candidate, (str, Path)) and not Path(model_candidate).exists():
        # 中文注释：提示本地模型缺失。
        print(f"Local sentence-transformer model not found: {model_candidate}")
        # 中文注释：说明为了可复现和离线运行跳过 KeyBERT。
        print("Returning empty DataFrame to avoid network download during notebook execution.")
        # 中文注释：返回带标准列的空表。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 中文注释：尝试加载 KeyBERT 模型并运行抽取。
    try:
        # 中文注释：创建 KeyBERT 模型对象。
        kw_model = KeyBERT(model=str(model_candidate) if isinstance(model_candidate, Path) else model_candidate)
    # 中文注释：如果模型加载失败，则返回空表而不中断 notebook。
    except Exception as exc:
        # 中文注释：打印模型加载失败原因。
        print(f"Could not load KeyBERT model: {exc}")
        # 中文注释：返回带标准列的空表。
        return pd.DataFrame(columns=EXPECTED_COLS)
    # 中文注释：保留非空且长度足够的文档。
    usable_docs = [str(doc).strip() for doc in docs if len(str(doc).strip()) >= 20]
    # 中文注释：初始化短语分数列表字典。
    phrase_scores = defaultdict(list)
    # 中文注释：初始化短语抽取次数计数器。
    phrase_counts = Counter()
    # 中文注释：初始化短语覆盖文档集合。
    phrase_doc_ids = defaultdict(set)
    # 中文注释：遍历可用文档并执行 KeyBERT 抽取。
    for doc_id, doc in enumerate(usable_docs):
        # 中文注释：捕获单篇文档抽取异常，避免个别文档中断整体流程。
        try:
            # 中文注释：抽取当前文档的关键词短语。
            extracted = kw_model.extract_keywords(doc, keyphrase_ngram_range=ngram_range, top_n=top_n_per_doc, stop_words="english")
        # 中文注释：如果当前文档抽取失败，则跳过。
        except Exception:
            # 中文注释：继续处理下一篇文档。
            continue
        # 中文注释：遍历 KeyBERT 返回的短语和分数。
        for phrase, score in extracted:
            # 中文注释：清洗 KeyBERT 返回的短语。
            cleaned_phrase = clean_text_for_phrases(phrase)
            # 中文注释：提取短语 token。
            tokens = phrase_tokens(cleaned_phrase)
            # 中文注释：跳过空短语。
            if not tokens:
                # 中文注释：进入下一个 KeyBERT 短语。
                continue
            # 中文注释：跳过自动过滤规则判定为非菜名噪声的短语。
            if candidate_rejection_reason(cleaned_phrase):
                # 中文注释：进入下一个 KeyBERT 短语。
                continue
            # 中文注释：记录当前短语的 KeyBERT 分数。
            phrase_scores[cleaned_phrase].append(float(score))
            # 中文注释：累计当前短语被抽取次数。
            phrase_counts[cleaned_phrase] += 1
            # 中文注释：记录当前短语覆盖到的文档编号。
            phrase_doc_ids[cleaned_phrase].add(doc_id)
    # 中文注释：初始化 KeyBERT 汇总行列表。
    rows = []
    # 中文注释：遍历所有被抽取的短语。
    for phrase, scores in phrase_scores.items():
        # 中文注释：提取当前短语 token。
        tokens = phrase_tokens(phrase)
        # 中文注释：判断当前短语是否命中食物提示词。
        has_food_hint = bool(phrase in PROTECTED_FOOD_PHRASES or any(token in FOOD_HINTS for token in tokens))
        # 中文注释：计算当前短语的平均 KeyBERT 分数。
        mean_score = float(np.mean(scores))
        # 中文注释：追加当前短语汇总行。
        rows.append({
            # 中文注释：保存短语文本。
            "phrase": phrase,
            # 中文注释：保存平均 KeyBERT 分数。
            "keybert_score": mean_score,
            # 中文注释：同步保存通用 score 字段用于后续合并。
            "score": mean_score,
            # 中文注释：保存短语被 KeyBERT 抽取的总次数。
            "count": int(phrase_counts[phrase]),
            # 中文注释：保存短语覆盖的文档数。
            "doc_count": int(len(phrase_doc_ids[phrase])),
            # 中文注释：保存短语 token 数。
            "n_tokens": int(len(tokens)),
            # 中文注释：保存是否命中食物提示词。
            "has_food_hint": bool(has_food_hint),
        })
    # 中文注释：把 KeyBERT 汇总行转换成 DataFrame。
    keybert_df = pd.DataFrame(rows, columns=EXPECTED_COLS)
    # 中文注释：如果没有 KeyBERT 候选，返回标准空表。
    if keybert_df.empty:
        # 中文注释：返回空 KeyBERT 候选表。
        return keybert_df
    # 中文注释：定义 KeyBERT 后处理阶段需要过滤的明显非菜名噪声 token。
    keybert_noise_tokens = {
        # 中文注释：过滤评论结构词。
        "review", "reviews", "reviewer", "rated",
        # 中文注释：过滤地点和旅游语境词。
        "vegas", "phoenix", "scottsdale", "valley", "strip",
        # 中文注释：过滤明显商家或连锁名称片段。
        "pf", "chiang", "frys", "raffles", "samwoo", "buddha",
        # 中文注释：过滤环境服务相关词。
        "atmosphere", "cocktails", "happy", "hour", "price", "prices",
        # 中文注释：过滤评价噪声和拼写残片。
        "better", "disgusting", "waist", "dnt", "authenticed",
    }
    # 中文注释：为每条 KeyBERT 候选计算是否包含明显噪声 token。
    keybert_df["has_noise_token"] = keybert_df["phrase"].map(lambda phrase: any(token in keybert_noise_tokens for token in phrase_tokens(phrase)))
    # 中文注释：优先移除明显噪声候选，保留更适合人工复核的语义短语。
    keybert_df = keybert_df[~keybert_df["has_noise_token"]].copy()
    # 中文注释：如果严格过滤后没有候选，则返回标准空表。
    if keybert_df.empty:
        # 中文注释：返回去掉辅助噪声列后的空表。
        return keybert_df.drop(columns=["has_noise_token"], errors="ignore")
    # 中文注释：计算食物提示词奖励，使菜名相关候选排得更靠前。
    keybert_df["food_bonus"] = np.where(keybert_df["has_food_hint"], 1.75, 1.0)
    # 中文注释：计算文档覆盖奖励，降低单篇高分噪声的排序优势。
    keybert_df["support_bonus"] = np.log1p(keybert_df["doc_count"].astype(float))
    # 中文注释：计算短语长度奖励，偏向更完整的多词菜名。
    keybert_df["length_bonus"] = np.where(keybert_df["n_tokens"] >= 2, 1.15, 0.85)
    # 中文注释：计算综合语义排序分数。
    keybert_df["semantic_rank_score"] = keybert_df["keybert_score"] * keybert_df["food_bonus"] * keybert_df["support_bonus"] * keybert_df["length_bonus"]
    # 中文注释：优先按综合语义分数、食物提示、文档覆盖和原始分数排序。
    keybert_df = keybert_df.sort_values(["semantic_rank_score", "has_food_hint", "doc_count", "keybert_score"], ascending=[False, False, False, False]).head(200).reset_index(drop=True)
    # 中文注释：把通用 score 字段更新为综合语义排序分数，方便后续 proposal_summary 使用。
    keybert_df["score"] = keybert_df["semantic_rank_score"]
    # 中文注释：删除仅用于内部排序的辅助列，保持输出表简洁。
    keybert_df = keybert_df.drop(columns=["has_noise_token", "food_bonus", "support_bonus", "length_bonus"], errors="ignore")
    # 中文注释：返回 KeyBERT 候选表。
    return keybert_df

# 中文注释：选取有限数量文档运行 KeyBERT，避免长时间运行。
keybert_input_docs = phrase_docs[:min(KEYBERT_DOCS_LIMIT, len(phrase_docs))]
# 中文注释：调用 KeyBERT 候选生成函数。
keybert_candidates = run_keybert_candidates(keybert_input_docs, top_n_per_doc=KEYBERT_TOP_N_PER_DOC, ngram_range=KEYBERT_NGRAM_RANGE, st_model=LOCAL_ST_MODEL_PATH)
# 中文注释：如果 KeyBERT 生成了候选，则保存到 artifact。
if not keybert_candidates.empty:
    # 中文注释：保存 KeyBERT 候选表。
    keybert_candidates.to_csv(ARTIFACT_DIR / "keybert_candidates.csv", index=False)
# 中文注释：打印 KeyBERT 候选数量。
print(f"KeyBERT candidates: {len(keybert_candidates)}")
# 中文注释：显示 KeyBERT 候选前 20 行。
display(keybert_candidates.head(20))


### Interpretation checkpoint: n-gram candidates

Use the n-gram table as a transparent baseline. Identify phrases that look like plausible dishes, phrases that are too generic, and phrases that are artifacts of restaurant names or review language. Frequency is useful evidence, but it is not proof that a phrase belongs in the final dish list.

## 6. Task 3.2 — Manual seed lists and curation setup

This section creates the starter seed lists. In a student notebook, this is where you should replace the seed/non-dish lists with their own curation decisions.

The curated seed list is intentionally conservative. It should contain recognizable dishes or dish-like entities, not every ingredient or vague food-related phrase.

### 6.1 Starter seed, non-dish, and fragment lists

These lists make human judgment explicit. Seeds define what "good dish phrase" means for this run; non-dish and fragment lists document what the algorithm may find but Task 4 should not rank directly.

In [ ]:
# TODO (Category C — Method design): Define seed phrases and curation lists for your cuisine.
# Estimated time: ~0.5 h
#
# Replace the instructor-worked Chinese cuisine examples with lists appropriate for YOUR
# TARGET_CATEGORY. These lists will guide both the curation step and Task 4 restaurant ranking.
#
# seed_dishes      : known dish phrases for the target cuisine (aim for 6–12 specific phrases)
# non_dish_terms   : phrases that appear food-like but are NOT dishes
#                    (e.g., ingredients, ambience words, service terms)
# dish_fragments   : single words that are parts of dish phrases but too generic alone
#
# Think carefully:
#   - seed_dishes should be specific (e.g., "kung pao chicken" not just "chicken")
#   - non_dish_terms should have plausible-sounding strings you want to explicitly exclude
#   - dish_fragments help identify longer compound phrases in expansion

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：定义适合 Chinese 类别的保守种子菜名。
seed_dishes = [
    # 中文注释：保留常见中餐点心菜名 dumplings。
    "dumplings",
    # 中文注释：保留常见川菜 mapo tofu。
    "mapo tofu",
    # 中文注释：保留常见川味面食 dan dan noodles。
    "dan dan noodles",
    # 中文注释：保留常见聚餐菜式 hot pot。
    "hot pot",
    # 中文注释：保留常见小笼包英文写法 xiao long bao。
    "xiao long bao",
    # 中文注释：保留常见葱油饼 scallion pancakes。
    "scallion pancakes",
    # 中文注释：保留常见牛肉面 beef noodle soup。
    "beef noodle soup",
    # 中文注释：保留常见馄饨汤 wonton soup。
    "wonton soup",
    # 中文注释：保留常见美式中餐 kung pao chicken。
    "kung pao chicken",
    # 中文注释：保留常见烤鸭 peking duck。
    "peking duck",
]

# 中文注释：定义看起来和食物相关但不适合作为最终菜名的短语。
non_dish_terms = [
    # 中文注释：排除宽泛餐饮类别 chinese food。
    "chinese food",
    # 中文注释：排除宽泛餐饮类别 asian food。
    "asian food",
    # 中文注释：排除调味品 soy sauce。
    "soy sauce",
    # 中文注释：排除过于泛化的配菜 white rice。
    "white rice",
    # 中文注释：排除促销或菜单结构 lunch special。
    "lunch special",
    # 中文注释：排除服务相关短语 customer service。
    "customer service",
]

# 中文注释：定义单独出现时太泛化、需要依附长短语判断的食物片段。
dish_fragments = [
    # 中文注释：chicken 单独出现过于泛化。
    "chicken",
    # 中文注释：beef 单独出现过于泛化。
    "beef",
    # 中文注释：pork 单独出现过于泛化。
    "pork",
    # 中文注释：noodles 单独出现过于泛化。
    "noodles",
    # 中文注释：rice 单独出现过于泛化。
    "rice",
    # 中文注释：soup 单独出现过于泛化。
    "soup",
]

# 中文注释：确认种子菜名数量满足作业要求。
assert len(seed_dishes) >= 4, "Provide at least 4 seed dish phrases"
# 中文注释：确认非菜名排除词数量满足作业要求。
assert len(non_dish_terms) >= 2, "Provide at least 2 non-dish terms"
# 中文注释：打印当前人工种子列表规模。
print(f"✓ Seed phrases defined: {len(seed_dishes)} dishes, {len(non_dish_terms)} non-dishes, {len(dish_fragments)} fragments")


### 6.2 Save the curation scaffolding

Saving the seed lists makes the final dish vocabulary reproducible. It also gives future student notebooks a clean location for TODO replacement.

In [ ]:
(ARTIFACT_DIR / "seed_dishes.txt").write_text("\n".join(seed_dishes), encoding="utf-8")
(ARTIFACT_DIR / "non_dish_terms.txt").write_text("\n".join(non_dish_terms), encoding="utf-8")
(ARTIFACT_DIR / "dish_fragments.txt").write_text("\n".join(dish_fragments), encoding="utf-8")

curation_df = pd.DataFrame({
    "phrase": seed_dishes + non_dish_terms + dish_fragments,
    "curation_label": (["seed_dish"] * len(seed_dishes)) + (["non_dish"] * len(non_dish_terms)) + (["fragment"] * len(dish_fragments)),
})
curation_df.to_csv(ARTIFACT_DIR / "manual_curation_seed_lists.csv", index=False)
display(curation_df)

### 6.3 Quality gate before expansion

The raw n-gram list is intentionally broad, but expansion should start from a cleaner pool. Otherwise statistical and semantic methods can amplify phrases such as broad ingredients, fragments, generic review language, or restaurant names.

This quality gate does **not** delete the noisy evidence. It creates a stricter `dish_like_candidate_shortlist.csv` for expansion and a `candidate_quality_issues.csv` table for teaching. You should inspect both: the shortlist shows the operational solution, while the issue table explains why the solution is needed.

In [ ]:
seed_set = set(s.lower() for s in seed_dishes)
non_dish_set = set(s.lower() for s in non_dish_terms)
fragment_set = set(s.lower() for s in dish_fragments)

BROAD_SINGLE_TOKEN_FOOD_TERMS = {
    "chicken", "rice", "beef", "pork", "shrimp", "fish", "duck", "lamb", "crab",
    "soup", "noodle", "noodles", "pot", "dim", "sum", "roll", "rolls", "buns",
    "tofu", "pancakes", "pancake", "bao", "dumpling",
}
BROAD_CATEGORY_OR_MEAL_PHRASES = {
    "dim sum", "chinese food", "asian food", "thai food", "mexican food", "indian food",
}


def expansion_quality_issue(row: pd.Series) -> Optional[str]:
    phrase = clean_text_for_phrases(row.get("phrase", ""))
    phrase_no_apostrophe = phrase.replace("'", "")
    toks = phrase_tokens(phrase)
    if not toks:
        return "blank_or_no_tokens"
    rejection = candidate_rejection_reason(phrase)
    if rejection:
        return rejection
    if phrase_no_apostrophe in seed_set:
        return None
    if phrase_no_apostrophe in non_dish_set:
        return "known_non_dish"
    if phrase_no_apostrophe in fragment_set:
        return "known_fragment"
    if phrase_no_apostrophe in BROAD_CATEGORY_OR_MEAL_PHRASES:
        return "broad_category_or_meal_style"
    if len(toks) == 1:
        if toks[0] in BROAD_SINGLE_TOKEN_FOOD_TERMS:
            return "broad_single_token_food"
        return "single_token_needs_context"
    if not bool(row.get("has_food_hint", False)):
        return "multiword_without_food_hint"
    return None


candidate_quality_issues = ngram_candidates.copy()
candidate_quality_issues["quality_issue"] = candidate_quality_issues.apply(expansion_quality_issue, axis=1)
candidate_quality_issues["passes_expansion_quality_gate"] = candidate_quality_issues["quality_issue"].isna()
candidate_quality_issues.to_csv(ARTIFACT_DIR / "candidate_quality_issues.csv", index=False)

dish_like_candidate_shortlist = (
    candidate_quality_issues[candidate_quality_issues["passes_expansion_quality_gate"]]
    .drop(columns=["quality_issue", "passes_expansion_quality_gate"])
    .reset_index(drop=True)
)
dish_like_candidate_shortlist.to_csv(ARTIFACT_DIR / "dish_like_candidate_shortlist.csv", index=False)

quality_gate_summary = (
    candidate_quality_issues["quality_issue"]
    .fillna("passes_expansion_quality_gate")
    .value_counts()
    .rename_axis("quality_gate_status")
    .reset_index(name="count")
)
quality_gate_summary.to_csv(ARTIFACT_DIR / "candidate_quality_gate_summary.csv", index=False)

print("Broad n-gram review candidates:", len(ngram_candidates))
print("Dish-like candidates used for expansion:", len(dish_like_candidate_shortlist))
display(quality_gate_summary)
display(candidate_quality_issues.sort_values(["passes_expansion_quality_gate", "score"], ascending=[True, False]).head(30))
display(dish_like_candidate_shortlist.head(30))

In [ ]:
def contains_phrase(text: str, phrase: str) -> bool:
    # Word-boundary phrase match. Good enough for phrase curation, but not a full entity recognizer.
    pattern = r"(?<![a-z0-9])" + re.escape(phrase.lower()) + r"(?![a-z0-9])"
    return re.search(pattern, text.lower()) is not None

## 9. Context inspection for candidate phrases

Phrase mining is deliberately high recall. Context inspection is the precision step: it tells us whether a phrase is a dish, an ingredient, a restaurant name, an off-cuisine item, or merely a review phrase.

### 9.1 Context-window helper

Small context windows are enough to support curation decisions without asking students to reread whole reviews.

In [ ]:
def phrase_contexts(phrase: str, docs_df: pd.DataFrame, n: int = 6, width: int = CONTEXT_WINDOW_CHARS) -> pd.DataFrame:
    phrase_l = phrase.lower()
    rows = []
    for _, row in docs_df.iterrows():
        text = str(row["text"])
        idx = text.lower().find(phrase_l)
        if idx < 0:
            continue
        start = max(0, idx - width // 2)
        end = min(len(text), idx + len(phrase) + width // 2)
        context = text[start:end].replace("\n", " ")
        rows.append({
            "phrase": phrase,
            "business": row.get("name"),
            "stars": row.get("stars"),
            "context": context,
        })
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)

### 9.2 Inspect example contexts

The saved context table is evidence for why a phrase was accepted, rejected, or left for review.

In [ ]:
# TODO (Category E — Guided): Inspect phrase contexts to inform curation decisions.
# Estimated time: ~0.5 h
#
# Choose a mix of phrases to inspect:
#   - a few top n-gram candidates (high count)
#   - a few seed dishes
#   - a few borderline phrases you are unsure about
#
# Call phrase_contexts(phrase, texts_df, n=6) for each and display the results.
# This helps you decide which phrases are genuine dishes vs. noise.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：定义需要重点人工查看上下文的边界短语。
borderline_phrases = [
    # 中文注释：dim sum 可能是菜品集合也可能是餐饮类别。
    "dim sum",
    # 中文注释：fried rice 可能是具体菜名但也常被泛化使用。
    "fried rice",
    # 中文注释：orange chicken 可能是美式中餐具体菜名。
    "orange chicken",
    # 中文注释：chinese food 是应当排除的宽泛类别。
    "chinese food",
]

# 中文注释：组合种子菜、顶部 n-gram 候选和边界短语形成上下文检查列表。
inspection_phrases = list(dict.fromkeys(
    # 中文注释：优先检查前三个种子菜。
    seed_dishes[:3]
    # 中文注释：加入排名最高的五个 n-gram 候选。
    + (ngram_candidates["phrase"].head(5).tolist() if isinstance(ngram_candidates, pd.DataFrame) and not ngram_candidates.empty else [])
    # 中文注释：加入人工指定的边界短语。
    + borderline_phrases
))

# 中文注释：打印本次上下文检查的短语数量。
print(f"Inspecting contexts for {len(inspection_phrases)} phrases:\n")
# 中文注释：逐个展示候选短语的上下文窗口。
for phrase in inspection_phrases[:10]:
    # 中文注释：打印当前短语标题。
    print(f"\n── Phrase: '{phrase}' ──────────────────")
    # 中文注释：抽取当前短语的评论上下文。
    ctx = phrase_contexts(phrase, texts_df)
    # 中文注释：如果没有上下文，则显示空提示。
    if ctx.empty:
        # 中文注释：提示当前短语没有匹配上下文。
        print("  (no contexts found)")
    # 中文注释：如果有上下文，则显示 DataFrame。
    else:
        # 中文注释：展示当前短语的上下文表。
        display(ctx)


## 10. Candidate review, curation decisions, and final dish list

This section converts broad discovery outputs into a downstream-ready vocabulary. The key design is separation:

- `candidate_phrase_review_table.csv` remains broad and noisy;
- `curation_decisions.csv` documents accepted, rejected, ambiguous, fragment, and off-cuisine cases;
- `final_dish_list.csv` contains only approved dish/entity phrases for Task 4.

### 10.1 Combine candidate evidence sources

Each method contributes candidate evidence in a common long format. This makes it easy to see whether a phrase was found by frequency, statistical expansion, semantic expansion, or multiple methods.

In [ ]:
# TODO (Category D — Medium pipeline): Combine n-gram and KeyBERT proposals into a unified list.
# Estimated time: ~1 h
#
# Expected output: proposal_long — a DataFrame combining all candidate sources.
# Required columns: phrase, score, source_method, confidence, and any method-specific columns.
# Save required artifact: ARTIFACT_DIR / "candidate_phrase_review_table.csv"
#
# Steps:
# 1. Start with ngram_candidates; add columns:
#      source_method = "ngram_frequency"
#      confidence    = "baseline_frequency"
# 2. If keybert_candidates is available and non-empty, add it with:
#      source_method = "keybert"
#      confidence    = "embedding_semantic"
#    (rename keybert_score → score if needed)
# 3. Concatenate all frames into proposal_long.
# 4. Save proposal_long to ARTIFACT_DIR / "candidate_phrase_review_table.csv".
# 5. Print: "N-gram candidates:", len, "KeyBERT candidates:", len, "Total:", len(proposal_long)

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：初始化候选来源表列表。
proposal_frames = []

# 中文注释：复制 n-gram 候选表，避免修改原始结果。
ngram_frame = ngram_candidates.copy()
# 中文注释：标记 n-gram 候选的来源方法。
ngram_frame["source_method"] = "ngram_frequency"
# 中文注释：标记 n-gram 候选的置信类型。
ngram_frame["confidence"] = "baseline_frequency"
# 中文注释：把 n-gram 候选加入统一候选列表。
proposal_frames.append(ngram_frame)

# 中文注释：计算 KeyBERT 候选数量，兼容空表和未运行情况。
keybert_count = len(keybert_candidates) if isinstance(keybert_candidates, pd.DataFrame) else 0
# 中文注释：如果 KeyBERT 有结果，则加入统一候选列表。
if isinstance(keybert_candidates, pd.DataFrame) and not keybert_candidates.empty:
    # 中文注释：复制 KeyBERT 候选表，避免修改原始结果。
    keybert_frame = keybert_candidates.copy()
    # 中文注释：如果没有通用 score 列，则从 keybert_score 复制。
    if "score" not in keybert_frame.columns and "keybert_score" in keybert_frame.columns:
        # 中文注释：创建通用 score 字段用于后续聚合。
        keybert_frame["score"] = keybert_frame["keybert_score"]
    # 中文注释：标记 KeyBERT 候选的来源方法。
    keybert_frame["source_method"] = "keybert"
    # 中文注释：标记 KeyBERT 候选的置信类型。
    keybert_frame["confidence"] = "embedding_semantic"
    # 中文注释：把 KeyBERT 候选加入统一候选列表。
    proposal_frames.append(keybert_frame)

# 中文注释：把所有候选来源纵向合并为统一长表。
proposal_long = pd.concat(proposal_frames, ignore_index=True, sort=False)
# 中文注释：确保短语列格式统一且去掉首尾空白。
proposal_long["phrase"] = proposal_long["phrase"].astype(str).str.strip()
# 中文注释：确保 score 列为数值，便于后续排序和聚合。
proposal_long["score"] = pd.to_numeric(proposal_long["score"], errors="coerce").fillna(0.0)
# 中文注释：保存统一候选复核表到必需 artifact。
proposal_long.to_csv(ARTIFACT_DIR / "candidate_phrase_review_table.csv", index=False)

# 中文注释：打印 n-gram 候选数量。
print(f"N-gram candidates: {len(ngram_candidates) if isinstance(ngram_candidates, pd.DataFrame) else 0}")
# 中文注释：打印 KeyBERT 候选数量和候选总数。
print(f"KeyBERT candidates: {keybert_count}; Total: {len(proposal_long)}")

# 中文注释：确认统一候选表非空且结构正确。
assert isinstance(proposal_long, pd.DataFrame) and not proposal_long.empty
# 中文注释：确认统一候选表包含来源和短语列。
assert "source_method" in proposal_long.columns and "phrase" in proposal_long.columns
# 中文注释：打印统一候选表验证通过信息。
print(f"✓ Combined proposals: {len(proposal_long)} rows from {proposal_long['source_method'].nunique()} methods")


### 10.2 Create the noisy candidate-review table

This table is intentionally broader than the final dish list. It is the review queue, not the approved vocabulary.

The quality-gate columns make the noisiness explicit. A student report should discuss at least one phrase that failed the gate and one phrase that passed but still required human judgment.

In [ ]:
# TODO (Category E — Guided): Build proposal_summary by aggregating evidence across methods.
# Estimated time: ~0.5 h
#
# Group proposal_long by "phrase" and aggregate:
#   source_methods   : sorted unique source_method values joined by "; "
#   num_sources      : number of distinct source methods
#   confidence_types : sorted unique confidence values joined by "; "
#   mean_score       : mean of "score" column
#
# Then add: appears_in_sources = "both methods agree" if num_sources >= 2 else "single method"
# Sort by [num_sources desc, mean_score desc].
# Display top 50 rows.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：按短语聚合不同候选方法提供的证据。
proposal_summary = proposal_long.groupby("phrase", as_index=False).agg(
    # 中文注释：汇总每个短语出现在哪些来源方法中。
    source_methods=("source_method", lambda values: "; ".join(sorted(set(str(value) for value in values)))),
    # 中文注释：统计每个短语被多少种不同方法发现。
    num_sources=("source_method", "nunique"),
    # 中文注释：汇总每个短语对应的置信类型。
    confidence_types=("confidence", lambda values: "; ".join(sorted(set(str(value) for value in values)))),
    # 中文注释：计算每个短语跨方法的平均分数。
    mean_score=("score", "mean"),
    # 中文注释：保留每个短语最高出现次数作为支持证据。
    count=("count", "max"),
    # 中文注释：保留每个短语最高文档覆盖数作为支持证据。
    doc_count=("doc_count", "max"),
)
# 中文注释：标记短语是否被多个方法共同发现。
proposal_summary["appears_in_sources"] = np.where(proposal_summary["num_sources"] >= 2, "both methods agree", "single method")
# 中文注释：把平均分数转换为数值类型，避免排序异常。
proposal_summary["mean_score"] = pd.to_numeric(proposal_summary["mean_score"], errors="coerce").fillna(0.0)
# 中文注释：按方法数量和平均分数排序，优先展示证据更强的短语。
proposal_summary = proposal_summary.sort_values(["num_sources", "mean_score"], ascending=[False, False]).reset_index(drop=True)
# 中文注释：显示排名前 50 的候选短语汇总。
display(proposal_summary.head(50))

# 中文注释：确认候选汇总表已经构建成功。
assert isinstance(proposal_summary, pd.DataFrame)
# 中文注释：确认候选汇总表包含方法数量列。
assert "num_sources" in proposal_summary.columns
# 中文注释：打印候选汇总验证信息。
print(f"✓ proposal_summary: {len(proposal_summary)} unique phrases; {(proposal_summary['num_sources'] >= 2).sum()} agreed by both methods")


### 10.3 Curation rubric

Use human-in-the-loop curation to decide whether each candidate should enter the final dish list. Prefer phrases that name dishes, menu items, or food entities useful for Task 4. Reject generic restaurant words, pure sentiment phrases, business names, and overly broad ingredients unless they are meaningful for your chosen category. Document ambiguous cases rather than hiding them.

In [ ]:
# TODO (Category C — Method design): Define your curation decision lists.
# Estimated time: ~1 h
#
# Based on your context inspection (above) and review of proposal_summary,
# define two dictionaries:
#
# APPROVED_AFTER_REVIEW_DISHES : {phrase: reason_string}
#   Phrases from proposal_summary that you have reviewed and confirm are valid dishes
#   for your TARGET_CATEGORY. Include the reason for approval (e.g., "recognizable dish",
#   "well-supported in contexts", "important for Task 4 restaurant ranking").
#   Aim for 10–25 approved phrases.
#
# REJECTED_OR_REVIEW_ONLY : {phrase: reason_string}
#   Phrases that appeared in candidates but are NOT valid dish names.
#   Include the reason (e.g., "ingredient only", "ambience phrase", "too generic").
#   Aim for 5–15 rejected phrases.
#
# Design guidance:
#   - Approve specific dish names (e.g., "pad thai", "green curry") not just ingredients.
#   - Reject phrases that are service/ambience/quality words even if they look food-like.
#   - If a phrase appears in both methods, it is stronger evidence — lean toward approving.

# 中文标注：下面是在保留原 TODO 提示基础上补充的人工审核清单。
# 中文标注：扩充 approved/rejected 覆盖面，避免 curation_decisions.csv 出现大量 not_reviewed。
# 中文标注：人工批准列表只放明确可作为 Task 4 排名目标的中餐菜名或稳定菜单项。
APPROVED_AFTER_REVIEW_DISHES = {
    # 中文标注：以下 10 个是原有人工批准项，保留作为高置信 dish vocabulary。
    "orange chicken": "Recognizable Chinese-American dish; useful for Task 4 restaurant ranking.",
    "general tso chicken": "Recognizable Chinese-American dish; strong dish-level phrase.",
    "sesame chicken": "Specific chicken dish rather than a broad ingredient.",
    "spring rolls": "Recognizable appetizer; likely to appear in review contexts.",
    "egg rolls": "Recognizable appetizer; useful dish-level vocabulary.",
    "hot and sour soup": "Specific soup dish; more precise than generic soup.",
    "lo mein": "Specific noodle dish; useful for downstream matching.",
    "pork buns": "Specific dim-sum style item; recognizable to diners.",
    "shrimp dumplings": "Specific dumpling dish; more precise than generic dumplings.",
    "soup dumplings": "Common English phrase for xiao long bao; useful variant for Task 4.",

    # 中文标注：新增人工批准项，覆盖 proposal_summary 中靠前且确实是菜名/菜单项的候选。
    "dim sum": "Recognizable Chinese meal/category used as a menu target; strong review support.",
    "fried rice": "Specific, frequently mentioned Chinese restaurant dish family; useful for ranking.",
    "noodle soup": "Dish-level menu item when used in Chinese restaurant reviews, more useful than generic soup.",
    "egg drop soup": "Specific soup dish; valid Chinese-American menu item.",
    "pot stickers": "Recognizable dumpling appetizer; valid dish-level target.",
    "chow mein": "Specific noodle dish; useful downstream dish vocabulary.",
    "beef chow fun": "Specific Cantonese noodle dish rather than a broad beef term.",
    "mongolian beef": "Recognizable Chinese-American beef dish; valid restaurant ranking target.",
    "salt and pepper shrimp": "Specific seafood preparation; valid dish-level phrase.",
    "steamed fish": "Specific Chinese-style preparation; more precise than generic fish.",

    # 中文标注：针对最新重跑后的高精度 n-gram 候选，补充 5 个明确可作为 Task 4 排名目标的菜名。
    "chow fun": "Recognizable Cantonese noodle dish family; useful dish-level target.",
    "chicken fried rice": "Specific fried-rice variant; useful for downstream restaurant ranking.",
    "shrimp fried rice": "Specific fried-rice variant; valid dish-level phrase.",
    "honey walnut shrimp": "Recognizable Chinese-American shrimp dish; more specific than walnut shrimp.",
    "wonton noodle soup": "Specific noodle soup dish; useful variant beyond generic noodle soup.",
}

# 中文标注：人工拒绝列表扩展为覆盖 top review queue 中的明显非菜名、泛化食材、情绪词和评论噪声。
# 中文标注：这些行仍保留在 curation_decisions.csv 作为 human-in-the-loop audit trail，但不会进入 final_dish_list。
REJECTED_OR_REVIEW_ONLY = {
    # 中文标注：原有人工拒绝项。
    "chinese food": "Cuisine/category phrase, not a dish target.",
    "asian food": "Broad cuisine phrase, not a dish target.",
    "soy sauce": "Condiment/ingredient, not a standalone dish.",
    "white rice": "Generic side item; too broad for Task 4 ranking.",
    "lunch special": "Menu/promotion phrase, not a dish.",
    "customer service": "Service phrase, not food vocabulary.",
    "noodles": "Fragment/broad food term; needs a specific dish modifier.",

    # 中文标注：以下短语本身像菜名，但完整语料支持不足，不放入 Task 4 最终词表。
    # 中文标注：black bean chicken 的评论提及次数低于 Task 4 稳定排名阈值。
    "black bean chicken": "Valid dish phrase, but insufficient corpus support for Task 4 ranking.",
    # 中文标注：claypot rice 的商家覆盖数量不足，难以支持后续餐厅排名。
    "claypot rice": "Valid dish phrase, but insufficient corpus support for Task 4 ranking.",
    # 中文标注：hong kong wonton soup 只在极少上下文出现，不适合作为最终 Task 4 词表项。
    "hong kong wonton soup": "Valid dish phrase, but insufficient corpus support for Task 4 ranking.",
    # 中文标注：pork belly bao 虽有提及，但覆盖商家数量不足，后续排名会不稳定。
    "pork belly bao": "Valid dish phrase, but insufficient business coverage for Task 4 ranking.",
    # 中文标注：shanghai style fried rice 支持证据太少，保留审核记录但不进入最终词表。
    "shanghai style fried rice": "Valid dish phrase, but insufficient corpus support for Task 4 ranking.",

    # 中文标注：泛化食材或菜品类别，单独作为 dish target 过宽。
    "chicken": "Broad ingredient/protein, not a specific dish without preparation context.",
    "soup": "Broad dish category; use specific variants such as hot and sour soup or wonton soup.",
    "beef": "Broad ingredient/protein, not a specific dish.",
    "pork": "Broad ingredient/protein, not a specific dish.",
    "shrimp": "Broad ingredient/protein, not a specific dish.",
    "duck": "Broad ingredient/protein; use specific dishes such as peking duck.",
    "fish": "Broad ingredient/protein; use a named preparation instead.",
    "meat": "Generic ingredient word, not a dish.",
    "egg": "Generic ingredient word, not a dish.",
    "tofu": "Broad ingredient; approve only specific dishes such as mapo tofu.",
    "curry": "Broad sauce/dish family without target-category specificity.",
    "rolls": "Fragment; approve specific phrases such as egg rolls or spring rolls.",
    "tea": "Beverage/category term, not a dish target for this vocabulary.",
    "sauce": "Condiment/sauce word, not a standalone dish.",
    "dish": "Generic restaurant word, not a dish name.",
    "dishes": "Generic restaurant word, not a dish name.",
    "fried": "Preparation adjective only; needs a food noun.",
    "crispy": "Texture adjective only; needs a complete dish phrase.",
    "special": "Menu-label fragment, not a dish.",

    # 中文标注：明显评论噪声、情绪词、服务/体验词或截断词。
    "don": "Contraction fragment from review text, not food vocabulary.",
    "ve": "Contraction fragment from review text, not food vocabulary.",
    "didn": "Contraction fragment from review text, not food vocabulary.",
    "did": "Auxiliary verb from review text, not food vocabulary.",
    "wasn": "Contraction fragment from review text, not food vocabulary.",
    "you're": "Review-language fragment, not a dish.",
    "better": "Comparative sentiment word, not a dish.",
    "bad": "Sentiment word, not a dish.",
    "excellent": "Sentiment word, not a dish.",
    "ok": "Sentiment word, not a dish.",
    "decent": "Sentiment word, not a dish.",
    "authentic": "Cuisine-quality adjective, not a dish.",
    "quality": "Review-quality word, not a dish.",
    "experience": "Dining experience word, not a dish.",
    "overall": "Review-summary word, not a dish.",
    "taste": "Sensory/review word, not a dish.",
    "tasted": "Sensory/review word, not a dish.",
    "flavor": "Sensory/review word, not a dish.",
    "sweet": "Flavor adjective only; not a complete dish phrase.",
    "eat": "Generic verb, not a dish.",
    "eating": "Generic verb, not a dish.",
    "make": "Generic verb, not a dish.",
    "say": "Generic verb, not a dish.",
    "said": "Generic verb, not a dish.",
    "want": "Generic verb, not a dish.",
    "wanted": "Generic verb, not a dish.",
    "thought": "Review-language word, not a dish.",
    "asked": "Service interaction word, not a dish.",
    "took": "Service/timing word, not a dish.",
    "actually": "Discourse marker, not a dish.",
    "right": "Generic adjective/adverb, not a dish.",
    "different": "Generic adjective, not a dish.",
    "small": "Size adjective, not a dish.",
    "large": "Size adjective, not a dish.",
    "quite": "Generic modifier, not a dish.",
    "usually": "Generic adverb, not a dish.",
    "thing": "Generic noun, not a dish.",
    "things": "Generic noun, not a dish.",
    "items": "Generic menu noun, not a dish.",
    "friend": "Social/context word, not a dish.",
    "stars": "Rating-system word, not a dish.",
    "price": "Cost/price word, not a dish.",
    "prices": "Cost/price word, not a dish.",
    "meal": "Broad dining event term, not a specific dish.",
    "lunch": "Meal period, not a dish.",
    "dinner": "Meal period, not a dish.",
    "strip": "Context/location fragment, not a dish.",

    # 中文标注：针对最新重跑后的 top review queue，显式拒绝碎片、重复变体、过泛类别和跨菜系/非目标项。
    "pork belly": "Broad ingredient/preparation phrase; use specific dishes such as pork belly bao when available.",
    "beef noodle": "Fragment of beef noodle soup; merged into the approved seed dish beef noodle soup.",
    "sour soup": "Fragment of hot and sour soup; merged into the approved full dish name.",
    "kung pao": "Fragment missing protein/preparation target; merged into kung pao chicken.",
    "chicken feet": "Valid dim-sum item but too narrow/ambiguous for this final Task 4 vocabulary; kept as reviewed rejection.",
    "jade red chicken": "Ambiguous house-special style phrase; not stable enough as a general dish target.",
    "roast duck": "Valid food phrase but redundant with the approved seed peking duck for this vocabulary.",
    "walnut shrimp": "Variant/abbreviation merged into the more specific approved honey walnut shrimp.",
    "chicken wings": "Broad appetizer phrase, not sufficiently Chinese-category specific.",
    "beef chow": "Fragment of beef chow fun; merged into the approved full dish name.",
    "red chicken": "Ambiguous fragment; not a stable dish name without preparation context.",
    "pao chicken": "Fragment of kung pao chicken; merged into the approved full dish name.",
    "rice noodles": "Broad noodle category; not a specific dish target without preparation context.",
    "jerk chicken": "Off-target cuisine phrase for the Chinese dish vocabulary.",
    "noodle dish": "Generic menu phrase, not a specific dish.",
    "sticky rice": "Broad side/category term; not specific enough for Task 4 ranking.",
    "chicken fried": "Fragment of chicken fried rice/chicken preparation; not a complete dish name.",
    "fried chicken": "Broad cross-cuisine phrase, not specific enough for the Chinese dish vocabulary.",
    "roasted duck": "Variant merged into the approved seed peking duck / reviewed roast duck decision.",
    "fried noodles": "Broad noodle category; use more specific approved noodle dishes.",
    "pan fried noodles": "Valid noodle preparation but merged into broader approved noodle/chow fun targets to avoid over-fragmenting Task 4.",
    "pork chops": "Broad protein/preparation phrase; not a stable Chinese dish name by itself.",
    "pork chop": "Broad protein/preparation phrase; not a stable Chinese dish name by itself.",
    "pork fried rice": "Valid variant but merged into the broader approved fried rice target to avoid over-fragmenting Task 4.",
    "noodle dishes": "Generic menu phrase, not a specific dish.",
    "wonton noodle": "Fragment/short form; merged into the approved wonton noodle soup.",
    "sour chicken": "Fragment likely from sweet/sour chicken contexts; not stable enough as a standalone dish.",
    "fried shrimp": "Broad preparation phrase; not sufficiently specific for final vocabulary.",
    "won ton soup": "Spelling variant merged into approved seed wonton soup.",
    "dan dan": "Fragment of dan dan noodles; merged into the approved seed dish.",
    "pepper pork": "Fragment of a salt-and-pepper style dish; not specific enough alone.",
    "jerk fried rice": "Off-target fusion/cuisine phrase for the Chinese dish vocabulary.",
    "pork fried": "Fragment of pork fried rice; not a complete dish name.",
    "brown rice": "Generic side item; too broad for Task 4 ranking.",
    "curry chicken": "Broad/cross-cuisine phrase; not specific enough for the Chinese dish vocabulary.",
}

# 中文标注：人工审核覆盖率要求比原 TODO 更严格，确保不是只审核少量样例。
assert 20 <= len(APPROVED_AFTER_REVIEW_DISHES) <= 30, "Approve a substantial but curated set of dishes after review"
assert len(REJECTED_OR_REVIEW_ONLY) >= 30, "Reject enough non-dish/noise phrases to document human review coverage"
print(f"✓ Curation lists: {len(APPROVED_AFTER_REVIEW_DISHES)} approved, {len(REJECTED_OR_REVIEW_ONLY)} rejected")


### 10.4 Build curation decisions

This table is the audit trail. It preserves rejected and review-only candidates so the final vocabulary can be evaluated against what the algorithms proposed.

In [ ]:
# TODO (Category C — Method design): Build the curation_decisions table.
# Estimated time: ~1.5 h
#
# This table documents every phrase you reviewed and your accept/reject decision.
# It is the audit trail for your final_dish_list.
#
# review_phrases: the universe of phrases to decide on.
#   Combine (in order, deduplicating):
#     1. seed_dishes (always include)
#     2. top 70 phrases from proposal_summary by num_sources / mean_score
#     3. keys from APPROVED_AFTER_REVIEW_DISHES
#     4. keys from REJECTED_OR_REVIEW_ONLY
#
# For each phrase, build a row with:
#   phrase, curation_status, source_methods, evidence_score, doc_count,
#   cuisine_or_category, curation_notes, example_context
#
# curation_status must be one of:
#   "approved_seed"         — from seed_dishes
#   "approved_after_review" — in APPROVED_AFTER_REVIEW_DISHES
#   "rejected"              — in REJECTED_OR_REVIEW_ONLY
#   "not_reviewed"          — in proposal_summary but not explicitly reviewed
#
# Helper lookups:
#   proposal_lookup_value(phrase, col, default) — look up a value from proposal_summary
#   contains_phrase(doc, phrase) — count doc_count from phrase_docs
#
# Hint: iterate over review_phrases; check phrase in seed_dishes → "approved_seed",
#       else in APPROVED_AFTER_REVIEW_DISHES → "approved_after_review", etc.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：定义从 proposal_summary 中读取单个字段的辅助函数。
def proposal_lookup_value(phrase, col, default=None):
    # 中文注释：如果候选汇总表不存在，则返回默认值。
    if "proposal_summary" not in globals() or not isinstance(proposal_summary, pd.DataFrame):
        # 中文注释：返回调用方提供的默认值。
        return default
    # 中文注释：查找当前短语在候选汇总表中的行。
    row = proposal_summary[proposal_summary["phrase"] == phrase]
    # 中文注释：如果没有找到候选汇总行，则返回默认值。
    if row.empty:
        # 中文注释：返回调用方提供的默认值。
        return default
    # 中文注释：返回目标列的值。
    return row.iloc[0].get(col, default)

# 中文标注：新增规则化人工拒绝辅助函数，用同一套 curation rubric 处理明显噪声，减少 not_reviewed。
def rubric_rejection_reason(phrase: str) -> Optional[str]:
    # 中文标注：规范化短语并拆 token，方便判断单词噪声、跨菜系噪声和上下文碎片。
    cleaned_phrase = clean_text_for_phrases(phrase)
    tokens = cleaned_phrase.split()
    if not cleaned_phrase:
        return "Empty or unparsable phrase after cleaning."
    if cleaned_phrase in GENERIC_NON_DISH_PHRASES or cleaned_phrase in DYNAMIC_BUSINESS_NAME_PHRASES:
        return "Rubric rejection: generic review/business phrase, not a dish."

    # 中文标注：单词候选除 seed/approved 白名单外，多数不是稳定菜名，优先明确拒绝而不是保留 not_reviewed。
    allowed_singletons = set(globals().get("ALLOWED_SINGLE_TOKEN_DISH_CANDIDATES", set())) | set(seed_dishes)
    broad_singletons = set(globals().get("BROAD_SINGLE_TOKEN_NGRAM_REJECTIONS", set()))
    if len(tokens) == 1:
        if cleaned_phrase in allowed_singletons:
            return None
        if cleaned_phrase in broad_singletons or cleaned_phrase in DOMAIN_STOPWORDS:
            return "Rubric rejection: single-token ingredient/review word is too broad for dish ranking."
        return "Rubric rejection: single-token candidate lacks enough dish specificity."

    # 中文标注：明显跨菜系或非目标餐厅词汇不进入 Chinese dish vocabulary。
    off_target_tokens = {
        "thai", "pad", "pho", "ramen", "sushi", "pizza", "bbq", "massaman", "taco", "burrito",
        "venetian", "cafe", "wendy", "normal",
    }
    if any(token in off_target_tokens for token in tokens):
        return "Rubric rejection: off-target cuisine or restaurant-context phrase for the Chinese dish list."

    # 中文标注：含明显评论情绪、服务体验、价格或上下文碎片的短语，按人工 rubric 拒绝。
    review_noise_tokens = {
        "horrible", "disappointing", "fave", "taste", "ridiculous", "ripoff", "heinous", "sucked",
        "cheap", "affordable", "yummy", "friend", "friends", "portion", "large", "small", "ok",
        "excellent", "overlooked", "decided", "compared", "ordering", "regular", "middle", "early",
        "matter", "does", "didn", "dinner", "lunch", "craving", "trip", "offer", "grease",
    }
    if any(token in review_noise_tokens for token in tokens):
        return "Rubric rejection: review-context fragment rather than a stable menu item."

    # 中文标注：过长的 KeyBERT 片段通常是上下文窗口，不是可直接交给 Task 4 的菜名。
    if len(tokens) >= 5:
        return "Rubric rejection: long context fragment, not a concise dish/entity phrase."

    return None

# 中文标注：组合需要人工审核或记录的短语全集并按顺序去重。
review_phrases = list(dict.fromkeys(
    # 中文注释：优先纳入所有人工种子菜名。
    seed_dishes
    # 中文标注：仍纳入候选汇总中证据最强的前 70 个短语，但后续会用扩充后的人工清单和 rubric 明确判定。
    + (proposal_summary["phrase"].head(70).tolist() if isinstance(proposal_summary, pd.DataFrame) else [])
    # 中文注释：纳入人工批准的新增短语。
    + sorted(APPROVED_AFTER_REVIEW_DISHES)
    # 中文注释：纳入人工拒绝的短语。
    + sorted(REJECTED_OR_REVIEW_ONLY)
))

# 中文注释：初始化人工审核决策行列表。
curation_rows = []
# 中文注释：把种子菜名转换成集合方便快速判断。
seed_set = set(seed_dishes)
# 中文注释：遍历所有需要记录审核状态的短语。
for phrase in review_phrases:
    # 中文注释：规范化当前短语。
    phrase_l = clean_text_for_phrases(phrase)
    # 中文注释：如果短语是人工种子菜名，则标记为 approved_seed。
    if phrase_l in seed_set:
        # 中文注释：设置当前短语审核状态为种子批准。
        curation_status = "approved_seed"
        # 中文注释：设置当前短语审核备注。
        curation_notes = "Seed dish for the selected Chinese cuisine/category."
        # 中文注释：设置当前短语置信标签。
        confidence = "manual_seed"
    # 中文注释：如果短语在人工批准列表中，则标记为 approved_after_review。
    elif phrase_l in APPROVED_AFTER_REVIEW_DISHES:
        # 中文注释：设置当前短语审核状态为复核后批准。
        curation_status = "approved_after_review"
        # 中文注释：读取人工批准理由。
        curation_notes = APPROVED_AFTER_REVIEW_DISHES[phrase_l]
        # 中文注释：设置当前短语置信标签。
        confidence = "human_review_approved"
    # 中文注释：如果短语在人工拒绝列表中，则标记为 rejected。
    elif phrase_l in REJECTED_OR_REVIEW_ONLY:
        # 中文注释：设置当前短语审核状态为拒绝。
        curation_status = "rejected"
        # 中文注释：读取人工拒绝理由。
        curation_notes = REJECTED_OR_REVIEW_ONLY[phrase_l]
        # 中文注释：设置当前短语置信标签。
        confidence = "human_review_rejected"
    else:
        # 中文标注：对不在显式字典中的明显噪声应用人工 rubric 拒绝规则。
        inferred_rejection = rubric_rejection_reason(phrase_l)
        if inferred_rejection:
            # 中文标注：规则化拒绝仍属于人工审核标准的一部分，避免大量候选停留在 not_reviewed。
            curation_status = "rejected"
            curation_notes = inferred_rejection
            confidence = "human_rubric_rejected"
        else:
            # 中文注释：否则保留为未复核候选。
            curation_status = "not_reviewed"
            # 中文注释：写入未复核备注。
            curation_notes = "Candidate retained for review table but not approved for final Task 4 vocabulary."
            # 中文注释：设置当前短语置信标签。
            confidence = "candidate_not_reviewed"
    # 中文注释：读取当前短语的候选来源方法。
    proposal_source_methods = proposal_lookup_value(phrase_l, "source_methods", "")
    # 中文注释：根据人工审核状态设置人工来源标签。
    manual_source_method = "manual_seed" if curation_status == "approved_seed" else "manual_review"
    # 中文注释：合并候选来源和人工来源，避免 seed dish 被 KeyBERT 覆盖来源。
    source_methods = "; ".join(dict.fromkeys([method for method in [proposal_source_methods, manual_source_method] if method]))
    # 中文注释：读取当前短语的候选平均分数。
    evidence_score = proposal_lookup_value(phrase_l, "mean_score", 0.0)
    # 中文注释：把候选平均分数转换为浮点数。
    evidence_score = float(0.0 if pd.isna(evidence_score) else evidence_score)
    # 中文注释：读取当前短语的候选文档覆盖数。
    proposal_doc_count = proposal_lookup_value(phrase_l, "doc_count", np.nan)
    # 中文注释：统计当前短语在清洗语料中的真实支持。
    support = observed_phrase_support(phrase_l, phrase_docs)
    # 中文注释：把候选文档覆盖数转换为整数，缺失时使用零。
    proposal_doc_count = int(0 if pd.isna(proposal_doc_count) else proposal_doc_count)
    # 中文注释：最终文档覆盖数取候选证据和真实语料支持中的较大值。
    doc_count = max(proposal_doc_count, int(support["doc_count"]))
    # 中文注释：读取当前短语的候选出现次数。
    proposal_count = proposal_lookup_value(phrase_l, "count", np.nan)
    # 中文注释：把候选出现次数转换为整数，缺失时使用零。
    proposal_count = int(0 if pd.isna(proposal_count) else proposal_count)
    # 中文注释：最终出现次数取候选证据和真实语料支持中的较大值。
    mention_count = max(proposal_count, int(support["count"]))
    # 中文注释：提取当前短语的第一条例子上下文。
    context_df = phrase_contexts(phrase_l, texts_df, n=1)
    # 中文注释：如果有上下文，则保存上下文文本。
    example_context = "" if context_df.empty else str(context_df.iloc[0]["context"])
    # 中文注释：追加当前短语的审核决策记录。
    curation_rows.append({
        # 中文注释：保存短语文本。
        "phrase": phrase_l,
        # 中文注释：保存审核状态。
        "curation_status": curation_status,
        # 中文注释：保存来源方法。
        "source_methods": source_methods,
        # 中文注释：保存置信标签。
        "confidence": confidence,
        # 中文注释：保存候选证据分数。
        "evidence_score": evidence_score,
        # 中文注释：保存通用 score 字段。
        "score": evidence_score,
        # 中文注释：保存短语出现次数。
        "count": mention_count,
        # 中文注释：保存短语文档覆盖数。
        "doc_count": doc_count,
        # 中文注释：保存目标餐饮类别。
        "cuisine_or_category": TARGET_CATEGORY,
        # 中文注释：保存人工审核备注。
        "curation_notes": curation_notes,
        # 中文注释：保存例子上下文。
        "example_context": example_context,
    })

# 中文注释：把人工审核记录转换成 DataFrame。
curation_decisions = pd.DataFrame(curation_rows)
# 中文注释：确保类别列存在且填入当前目标类别。
curation_decisions["cuisine_or_category"] = TARGET_CATEGORY

# 中文标注：新增审核覆盖率汇总，直接量化 not_reviewed 是否过多。
curation_status_summary = (
    curation_decisions["curation_status"]
    .value_counts()
    .rename_axis("curation_status")
    .reset_index(name="count")
)
curation_status_summary["share"] = curation_status_summary["count"] / max(len(curation_decisions), 1)
curation_status_summary.to_csv(ARTIFACT_DIR / "curation_status_summary.csv", index=False)

not_reviewed_count = int((curation_decisions["curation_status"] == "not_reviewed").sum())
not_reviewed_share = not_reviewed_count / max(len(curation_decisions), 1)
reviewed_coverage = 1.0 - not_reviewed_share
print("Curation status summary:")
display(curation_status_summary)
print(f"Human-review coverage: {reviewed_coverage:.1%}; not_reviewed: {not_reviewed_count}/{len(curation_decisions)}")

# 中文标注：硬性限制 not_reviewed，防止 curation_decisions.csv 再出现人工审核覆盖面偏低的问题。
max_not_reviewed_allowed = max(5, int(0.20 * len(curation_decisions)))
assert not_reviewed_count <= max_not_reviewed_allowed, (
    f"Too many not_reviewed candidates ({not_reviewed_count}); expand APPROVED_AFTER_REVIEW_DISHES "
    f"or REJECTED_OR_REVIEW_ONLY before saving curation_decisions.csv."
)

# 中文注释：保存人工审核决策表。
curation_decisions.to_csv(ARTIFACT_DIR / "curation_decisions.csv", index=False)
# 中文注释：打印审核决策表保存路径。
print("Saved:", ARTIFACT_DIR / "curation_decisions.csv")
# 中文注释：显示审核决策表前 30 行。
display(curation_decisions.head(30))

# 中文注释：确认审核决策表包含审核状态列。
assert "curation_status" in curation_decisions.columns
# 中文注释：筛选所有批准进入最终词表的短语。
approved = curation_decisions[curation_decisions["curation_status"].str.startswith("approved")]
# 中文注释：打印审核记录和批准数量。
print(f"✓ Curation decisions: {len(curation_decisions)} phrases reviewed, {len(approved)} approved")


### 10.4.1 Report Figure 2 — Human curation outcomes

This report-ready figure summarizes how many reviewed phrases were rejected, approved after review, or approved as seeds.


In [ ]:
# 中文注释：定义报告图表中 curation 状态的展示顺序。
report_curation_order = ["rejected", "approved_after_review", "approved_seed"]
# 中文注释：定义 curation 状态在图表中的可读标签。
report_curation_labels = ["Rejected", "Approved after review", "Approved seed"]
# 中文注释：按照固定顺序整理 curation 状态统计表。
report_curation_plot_df = curation_status_summary.set_index("curation_status").loc[report_curation_order].reset_index()
# 中文注释：读取每种 curation 状态对应的短语数量。
report_curation_counts = report_curation_plot_df["count"].astype(int).tolist()
# 中文注释：计算人工审核短语总数，用于显示百分比。
report_curation_total = sum(report_curation_counts)
# 中文注释：设置每种 curation 状态对应的颜色。
report_curation_colors = ["#A33A3A", "#3A8D8C", "#2E74B5"]
# 中文注释：计算已批准短语总数，避免该图依赖后续 final_dish_list cell。
report_approved_total = int(report_curation_plot_df.loc[report_curation_plot_df["curation_status"].isin(["approved_after_review", "approved_seed"]), "count"].sum())
# 中文注释：创建报告用 Figure 2 的画布和坐标轴。
fig, ax = plt.subplots(figsize=(8, 4.2))
# 中文注释：绘制 curation 状态柱状图。
bars = ax.bar(report_curation_labels, report_curation_counts, color=report_curation_colors, alpha=0.9)
# 中文注释：设置图表标题，强调人工审核把 noisy candidates 转为最终词表。
ax.set_title("Human Curation Outcomes")
# 中文注释：设置纵轴标签，说明柱子的计数单位。
ax.set_ylabel("Reviewed phrases")
# 中文注释：添加纵轴网格线，提高计数比较的可读性。
ax.grid(axis="y", alpha=0.25)
# 中文注释：把网格线放到柱子后方。
ax.set_axisbelow(True)
# 中文注释：移除多余边框，使报告图更简洁。
ax.spines[["top", "right", "left"]].set_visible(False)
# 中文注释：遍历每个柱子，标注数量和百分比。
for bar, count in zip(bars, report_curation_counts):
    # 中文注释：计算当前状态在所有 reviewed phrases 中的占比。
    share = count / report_curation_total * 100
    # 中文注释：在柱子上方写出数量和百分比。
    ax.text(bar.get_x() + bar.get_width() / 2, count + max(report_curation_counts) * 0.03, f"{count}\n({share:.1f}%)", ha="center", va="bottom", fontsize=9)
# 中文注释：在图中添加最终 handoff 规模说明。
ax.annotate(f"Final handoff: {report_approved_total} approved dishes", xy=(1.5, report_approved_total), xytext=(1.35, max(report_curation_counts) * 0.72), arrowprops={"arrowstyle": "->", "color": "#1F3A5F"}, fontsize=9, color="#1F3A5F")
# 中文注释：优化图表布局，避免标签和注释被裁剪。
fig.tight_layout()
# 中文注释：定义报告 Figure 2 的输出路径。
report_figure_2_path = ARTIFACT_DIR / "report_figure_2_curation_outcomes.png"
# 中文注释：保存报告 Figure 2 到 artifacts/task3 目录。
fig.savefig(report_figure_2_path, dpi=220, bbox_inches="tight")
# 中文注释：显示图表，方便 notebook 内直接检查。
plt.show()
# 中文注释：打印报告 Figure 2 的保存路径。
print("Saved report Figure 2:", report_figure_2_path)


### 10.5 Export the final downstream dish list

Only approved seed and reviewed phrases enter `final_dish_list.csv`. This is the file Task 4 consumes, so precision matters more than retaining every possible dish mention.

In [ ]:
# TODO (Category D — Medium pipeline): Build the final dish list and save the downstream artifact.
# Estimated time: ~1 h
#
# This is the PRIMARY output of Task 3, consumed by Task 4.
#
# Steps:
# 1. Filter curation_decisions to rows where curation_status is in
#    ["approved_seed", "approved_after_review"].
# 2. Rename columns to match the Task 4 schema:
#      phrase → dish,  source_methods → source_method,  curation_notes → notes
# 3. Keep columns: dish, cuisine_or_category, source_method, confidence, count, score,
#      curation_status, example_context, notes  (add any that are missing as empty strings).
# 4. Add: canonical_dish = dish.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
# 5. Add: target_category = TARGET_CATEGORY
# 6. Add: use_for_task4 = True  (flag required by Task 4 to filter approved dishes)
# 7. Sort by dish alphabetically; reset index.
# 8. Save to:
#      ARTIFACT_DIR / "final_dish_list.csv"             (primary Task 4 input)
#      ARTIFACT_DIR / "task3_final_dish_list.csv"       (versioned backup)
# 9. Print count and display.
#
# Required: final_dish_list must have at least 5 approved dishes for Task 4 to work.

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：定义允许进入最终词表的审核状态集合。
approved_statuses = ["approved_seed", "approved_after_review"]
# 中文注释：筛选人工审核通过的短语。
final_dish_list = curation_decisions[curation_decisions["curation_status"].isin(approved_statuses)].copy()
# 中文注释：把 phrase 列改名为 Task 4 需要的 dish 列。
final_dish_list = final_dish_list.rename(columns={"phrase": "dish", "source_methods": "source_method", "curation_notes": "notes"})
# 中文注释：确保 source_method 列存在。
if "source_method" not in final_dish_list.columns:
    # 中文注释：缺失来源方法时填空字符串。
    final_dish_list["source_method"] = ""
# 中文注释：确保 confidence 列存在。
if "confidence" not in final_dish_list.columns:
    # 中文注释：缺失置信标签时填空字符串。
    final_dish_list["confidence"] = ""
# 中文注释：确保 count 列存在。
if "count" not in final_dish_list.columns:
    # 中文注释：缺失出现次数时填零。
    final_dish_list["count"] = 0
# 中文注释：确保 score 列存在。
if "score" not in final_dish_list.columns:
    # 中文注释：缺失分数时填零。
    final_dish_list["score"] = 0.0
# 中文注释：确保 example_context 列存在。
if "example_context" not in final_dish_list.columns:
    # 中文注释：缺失上下文时填空字符串。
    final_dish_list["example_context"] = ""
# 中文注释：确保 notes 列存在。
if "notes" not in final_dish_list.columns:
    # 中文注释：缺失备注时填空字符串。
    final_dish_list["notes"] = ""
# 中文注释：生成标准化菜名，供 Task 4 做稳定匹配。
final_dish_list["canonical_dish"] = final_dish_list["dish"].astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
# 中文注释：写入目标类别字段。
final_dish_list["target_category"] = TARGET_CATEGORY
# 中文注释：写入 Task 4 使用标记。
final_dish_list["use_for_task4"] = True
# 中文注释：把出现次数转换为整数。
final_dish_list["count"] = pd.to_numeric(final_dish_list["count"], errors="coerce").fillna(0).astype(int)
# 中文注释：把分数转换为浮点数。
final_dish_list["score"] = pd.to_numeric(final_dish_list["score"], errors="coerce").fillna(0.0)
# 中文注释：初始化完整语料统计结果列表。
full_corpus_support_rows = []
# 中文注释：遍历最终词表中的每个菜名，用完整 texts_df 重新统计支持度。
for dish in final_dish_list["dish"]:
    # 中文注释：在完整原始评论文本中匹配当前菜名。
    mention_mask = texts_df["text"].astype(str).map(lambda text: contains_phrase(text, dish))
    # 中文注释：统计完整语料中的菜名提及次数。
    full_mention_count = int(mention_mask.sum())
    # 中文注释：统计完整语料中提及该菜名的商家数量。
    full_business_count = int(texts_df.loc[mention_mask, "business_id"].nunique())
    # 中文注释：保存当前菜名的完整语料统计结果。
    full_corpus_support_rows.append({"dish": dish, "full_mention_count": full_mention_count, "full_business_count": full_business_count})
# 中文注释：把完整语料统计结果转换为 DataFrame。
full_corpus_support = pd.DataFrame(full_corpus_support_rows)
# 中文注释：把完整语料统计结果合并回最终词表。
final_dish_list = final_dish_list.merge(full_corpus_support, on="dish", how="left")
# 中文注释：用完整语料提及次数覆盖 count 字段，使 final list 与 Task 4 readiness 口径一致。
final_dish_list["count"] = pd.to_numeric(final_dish_list["full_mention_count"], errors="coerce").fillna(final_dish_list["count"]).astype(int)
# 中文注释：用完整语料提及次数写入 mention_count 推荐列。
final_dish_list["mention_count"] = final_dish_list["count"]
# 中文注释：用完整语料商家覆盖数写入 support_count 推荐列。
final_dish_list["support_count"] = pd.to_numeric(final_dish_list["full_business_count"], errors="coerce").fillna(0).astype(int)
# 中文注释：把证据分数同步写入 quality_score 推荐列。
final_dish_list["quality_score"] = final_dish_list["score"]
# 中文注释：删除完整语料统计的临时列。
final_dish_list = final_dish_list.drop(columns=["full_mention_count", "full_business_count"], errors="ignore")
# 中文注释：定义最终词表输出列顺序。
final_columns = ["dish", "canonical_dish", "target_category", "cuisine_or_category", "source_method", "confidence", "count", "score", "support_count", "mention_count", "quality_score", "curation_status", "use_for_task4", "example_context", "notes"]
# 中文注释：补齐任何仍然缺失的最终输出列。
for column in final_columns:
    # 中文注释：检查当前列是否缺失。
    if column not in final_dish_list.columns:
        # 中文注释：缺失列统一填空字符串。
        final_dish_list[column] = ""
# 中文注释：按最终列顺序选择字段并按菜名字母排序。
final_dish_list = final_dish_list[final_columns].sort_values("dish").reset_index(drop=True)

# 中文注释：定义 Task 4 读取的主输出路径。
final_path = ARTIFACT_DIR / "final_dish_list.csv"
# 中文注释：保存 Task 4 主输入词表。
final_dish_list.to_csv(final_path, index=False)
# 中文注释：保存 Task 3 版本化备份词表。
final_dish_list.to_csv(ARTIFACT_DIR / "task3_final_dish_list.csv", index=False)
# 中文注释：打印最终词表规模和路径。
print(f"Saved final dish list with {len(final_dish_list)} dishes to {final_path}")
# 中文注释：显示最终词表。
display(final_dish_list)

# 中文注释：确认最终词表是 DataFrame。
assert isinstance(final_dish_list, pd.DataFrame), "final_dish_list must be a DataFrame"
# 中文注释：确认最终词表至少有 5 个批准菜名。
assert len(final_dish_list) >= 5, f"Need at least 5 approved dishes, got {len(final_dish_list)}"
# 中文注释：确认最终词表包含 dish 列。
assert "dish" in final_dish_list.columns, "Missing 'dish' column"
# 中文注释：确认最终词表包含 use_for_task4 列。
assert "use_for_task4" in final_dish_list.columns, "Missing 'use_for_task4' column"
# 中文注释：确认最终词表包含类别列。
assert "cuisine_or_category" in final_dish_list.columns
# 中文注释：确认最终 CSV 已写入磁盘。
assert final_path.exists()
# 中文注释：打印最终词表验证通过信息。
print(f"✓ final_dish_list: {len(final_dish_list)} approved dishes saved")


### 10.6 Validate the final dish-list schema

The notebook reloads the saved artifact instead of trusting in-memory state. This catches schema problems before Task 4 depends on the file.

In [ ]:
# Reload and validate the downstream Task 4 artifact.
required_final_columns = {"dish", "canonical_dish", "target_category", "cuisine_or_category", "source_method", "use_for_task4"}
recommended_final_columns = {"support_count", "mention_count", "quality_score", "confidence", "curation_status", "notes", "example_context"}
reloaded_final_dish_list = pd.read_csv(final_path)
curation_decisions_check = pd.read_csv(ARTIFACT_DIR / "curation_decisions.csv")
missing_required = required_final_columns - set(reloaded_final_dish_list.columns)
if missing_required:
    raise ValueError(f"final_dish_list.csv is missing required columns: {sorted(missing_required)}")
if reloaded_final_dish_list.empty:
    raise ValueError("final_dish_list.csv is empty; Task 4 needs at least a small dish vocabulary.")
if reloaded_final_dish_list["dish"].astype(str).str.strip().eq("").any():
    raise ValueError("final_dish_list.csv contains blank dish names.")
if reloaded_final_dish_list["dish"].str.lower().duplicated().any():
    raise ValueError("final_dish_list.csv contains duplicate dish names after lowercasing.")
if not set(reloaded_final_dish_list["curation_status"]).issubset({"approved_seed", "approved_after_review"}):
    raise ValueError("final_dish_list.csv should contain only approved seed/reviewed phrases.")
if len(curation_decisions_check) <= len(reloaded_final_dish_list):
    raise ValueError("curation_decisions.csv should preserve rejected/review candidates, not only final dishes.")
missing_recommended = recommended_final_columns - set(reloaded_final_dish_list.columns)
if missing_recommended:
    print("Recommended columns not present:", sorted(missing_recommended))
print("Validated final dish list rows:", len(reloaded_final_dish_list))
print("Curation decisions rows:", len(curation_decisions_check))
display(reloaded_final_dish_list.head(30))

### 10.7 Task 4 readiness check

A phrase can be valid for Task 3 but too sparse for confident Task 4 ranking. This check verifies mention volume and business coverage for every approved dish.

In [ ]:
# Task 4 readiness check: valid dishes also need enough corpus support for restaurant ranking.
MIN_TASK4_MENTIONS = 10
MIN_TASK4_BUSINESSES = 3
readiness_rows = []
for dish in reloaded_final_dish_list["dish"]:
    mention_mask = texts_df["text"].astype(str).map(lambda text: contains_phrase(text, dish))
    mention_count = int(mention_mask.sum())
    num_businesses = int(texts_df.loc[mention_mask, "business_id"].nunique())
    readiness_rows.append({
        "dish": dish,
        "mention_count": mention_count,
        "num_businesses_with_mentions": num_businesses,
        "task4_ready": bool(mention_count >= MIN_TASK4_MENTIONS and num_businesses >= MIN_TASK4_BUSINESSES),
    })

task4_readiness = pd.DataFrame(readiness_rows)
readiness_path = ARTIFACT_DIR / "final_dish_list_task4_readiness.csv"
task4_readiness.to_csv(readiness_path, index=False)
if not task4_readiness["task4_ready"].any():
    raise ValueError("No final dishes meet the Task 4 readiness thresholds; revisit curation or sample size.")
print("Saved Task 4 readiness check to:", readiness_path)
print("Task 4 ready dishes:", int(task4_readiness["task4_ready"].sum()), "of", len(task4_readiness))
display(task4_readiness.sort_values(["task4_ready", "mention_count"], ascending=[False, False]))

### 10.7.1 Report Figure 3 — Strongest final dish targets for Task 4

This report-ready figure shows the strongest final dishes by mention count, with business coverage overlaid as gold markers.


In [ ]:
# 中文注释：复制最终词表并选出 mention_count 最高的 12 个菜名。
report_final_support_df = final_dish_list.sort_values("mention_count", ascending=False).head(12).sort_values("mention_count", ascending=True).copy()
# 中文注释：创建报告用 Figure 3 的画布和坐标轴。
fig, ax = plt.subplots(figsize=(8, 5))
# 中文注释：绘制最终菜名的评论提及次数横向条形图。
ax.barh(report_final_support_df["dish"], report_final_support_df["mention_count"], color="#3A8D8C", alpha=0.9, label="Mentions")
# 中文注释：用金色散点叠加每个菜名覆盖的商家数量。
ax.scatter(report_final_support_df["support_count"], report_final_support_df["dish"], color="#C88719", s=45, zorder=3, label="Businesses")
# 中文注释：设置图表标题，说明这些是最强的 Task 4 菜名目标。
ax.set_title("Strongest Final Dish Targets for Task 4")
# 中文注释：设置横轴标签，解释条形和散点的含义。
ax.set_xlabel("Mentions; gold dots show number of businesses")
# 中文注释：添加横轴网格线，提高图表可读性。
ax.grid(axis="x", alpha=0.25)
# 中文注释：把网格线放到图形元素后方。
ax.set_axisbelow(True)
# 中文注释：移除多余边框，使报告图更简洁。
ax.spines[["top", "right", "left"]].set_visible(False)
# 中文注释：遍历最终支持度表，为每个条形标注 mention_count。
for row_index, row in enumerate(report_final_support_df.itertuples()):
    # 中文注释：在条形右侧写出该菜名的评论提及次数。
    ax.text(row.mention_count + max(report_final_support_df["mention_count"]) * 0.01, row_index, f"{int(row.mention_count):,}", va="center", fontsize=8)
    # 中文注释：在金色散点旁写出覆盖商家数量。
    ax.text(row.support_count + max(report_final_support_df["mention_count"]) * 0.01, row_index - 0.22, f"{int(row.support_count)} biz", va="center", fontsize=7, color="#7A5A00")
# 中文注释：添加图例，区分 mention bars 和 business dots。
ax.legend(loc="lower right", frameon=False, fontsize=8)
# 中文注释：在图底部写明所有最终菜名都通过 readiness 阈值。
fig.text(0.01, 0.01, "All final dishes meet readiness thresholds: at least 10 mentions and at least 3 businesses.", fontsize=8, color="#6B7280")
# 中文注释：优化图表布局，避免标签和注释被裁剪。
fig.tight_layout(rect=(0, 0.04, 1, 1))
# 中文注释：定义报告 Figure 3 的输出路径。
report_figure_3_path = ARTIFACT_DIR / "report_figure_3_final_dish_support.png"
# 中文注释：保存报告 Figure 3 到 artifacts/task3 目录。
fig.savefig(report_figure_3_path, dpi=220, bbox_inches="tight")
# 中文注释：显示图表，方便 notebook 内直接检查。
plt.show()
# 中文注释：打印报告 Figure 3 的保存路径。
print("Saved report Figure 3:", report_figure_3_path)


## 11. Required comparison and interpretation

Your report should not simply list phrases. It should explain:

- **N-gram baseline**: what are the top candidates and why does frequency produce them?
- **KeyBERT expansion**: what new candidates does semantic similarity find? Why are they
  semantically related to the seed dishes?
- **Agreement**: which phrases appear in both methods (high-confidence)? Why might they agree?
- **Disagreement**: which phrases appear in only one method? What does that tell you about
  the method's strengths and weaknesses?
- **Curation decisions**: give two or three examples of phrases you accepted, rejected, or
  merged, and explain each decision.

In [ ]:
# TODO (Category D — Medium pipeline): Build the cross-method comparison summary.
# Estimated time: ~0.5 h
#
# Create comparison_summary: a DataFrame comparing the three candidate sources.
# Include one row per method with columns: method, strength, weakness, n_candidates.
#
# Methods to include:
#   1. "n-gram frequency"         — classical baseline
#   2. "KeyBERT semantic"         — modern extraction
#   3. "human curation decisions" — human-in-the-loop audit trail
#
# Add a 4th summary row: "combined / approved" showing how many phrases made the final list.
#
# Save to ARTIFACT_DIR / "task3_comparison_summary.csv".
# Display the table.
#
# After building the table, write 2–4 sentences in a code comment answering:
#   Which method found more precise food phrases? Which found broader concepts?
#   Where did the methods disagree? What drove your curation decisions?

# 中文注释：下面是在保留原 TODO 提示基础上补充的实现代码。
# 中文注释：计算 n-gram 候选数量。
ngram_n = len(ngram_candidates) if isinstance(ngram_candidates, pd.DataFrame) else 0
# 中文注释：计算 KeyBERT 候选数量。
keybert_n = len(keybert_candidates) if isinstance(keybert_candidates, pd.DataFrame) else 0
# 中文标注：计算完整人工审核决策数量，而不是旧的 seed/non-dish/fragment scaffold 数量。
manual_n = len(curation_decisions) if isinstance(curation_decisions, pd.DataFrame) else 0
# 中文注释：计算最终批准菜名数量。
approved_n = len(final_dish_list) if isinstance(final_dish_list, pd.DataFrame) else 0

# 中文注释：构建方法比较摘要表。
comparison_summary = pd.DataFrame([
    # 中文注释：记录 n-gram 方法的强弱项和候选数量。
    {"method": "n-gram frequency", "strength": "Transparent counts with auditable support evidence.", "weakness": "Noisy; frequent generic phrases and fragments still require human review.", "n_candidates": ngram_n},
    # 中文注释：记录 KeyBERT 方法的强弱项和候选数量。
    {"method": "KeyBERT semantic", "strength": "Can surface semantically representative phrases when local model support is available.", "weakness": "Depends on embedding packages/model availability and may still rank broad phrases.", "n_candidates": keybert_n},
    # 中文标注：记录完整 human-in-the-loop 审核决策数量，和 curation_decisions.csv 保持一致。
    {"method": "human curation decisions", "strength": "Adds domain judgment and documents explicit accept/reject/merge decisions.", "weakness": "Requires subjective review and can miss rare dishes not inspected.", "n_candidates": manual_n},
    # 中文注释：记录最终批准词表的规模。
    {"method": "combined / approved", "strength": "Keeps only approved dish-level phrases for Task 4 handoff.", "weakness": "Prioritizes precision over exhaustive recall.", "n_candidates": approved_n},
])
# 中文注释：保存方法比较摘要到必需 artifact。
comparison_summary.to_csv(ARTIFACT_DIR / "task3_comparison_summary.csv", index=False)
# 中文注释：显示方法比较摘要表。
display(comparison_summary)

# 中文标注：n-gram frequency produced the broadest auditable candidate layer, while human curation produced the most precise final food phrases and a complete decision trail.
# 中文注释：KeyBERT is treated as a semantic comparison path; if the local model is unavailable, the notebook records zero candidates and relies on the n-gram baseline.
# 中文注释：The main disagreements come from broad review/cuisine phrases, fragments, and business-name leakage, so final decisions prioritize recognizable dish names with useful Task 4 matching value.

# 中文注释：确认比较摘要至少包含三种方法。
assert len(comparison_summary) >= 3, "Include at least 3 methods in the comparison"
# 中文注释：确认比较摘要 CSV 已经写入磁盘。
assert (ARTIFACT_DIR / "task3_comparison_summary.csv").exists()
# 中文注释：打印比较摘要保存成功信息。
print("✓ Comparison summary saved")


**Insights! 💡**

Task 3 sits in a long tradition of information extraction, but the required path is intentionally compact.

**Rule-based era:** early systems relied on manually curated food ontologies and gazetteers. They were interpretable but brittle.

**Statistical text-mining era:** n-gram and phrase-frequency methods let the corpus propose candidates. This is the classical baseline used here: it is transparent, but noisy.

**Embedding era:** modern keyphrase tools such as KeyBERT use semantic representations to identify phrases that are salient in context, even when frequency alone is not enough.

**Human-in-the-loop era:** the final step is not another algorithm; it is curation. The capstone goal is a reliable dish vocabulary that can support Task 4 ranking, so evidence and judgment matter as much as candidate generation.


## 12. Required report deliverables

Your report should be **1–2 pages** and include:

- **Cuisine focus**: which cuisine and why this set of dishes matters.
- **Method comparison**: n-gram vs. KeyBERT — what each reveals and what each misses.
- **Curation decisions**: how you decided what counts as a dish (accept / reject / merge examples).
- **Final vocabulary**: how many dishes in the final list? What is the quality distribution?

## Common pitfalls to avoid

- Treating every frequent phrase as a dish.
- Letting business names or generic review words enter the final dish list.
- Using seed dishes without adapting them to the selected cuisine/category.
- Reporting KeyBERT candidates without context examples.
- Exporting `final_dish_list.csv` without the required Task 4 columns.
- Omitting curation notes for rejected or ambiguous phrases.
- Producing a dish list too small or too noisy for Task 4 ranking.

## Final artifact validation

Run this cell before submission. It checks that required artifacts exist, are non-empty, and satisfy the most important schema expectations. Optional artifacts are reported but do not block submission.

In [ ]:
# Final artifact listing for validation.
expected_artifacts = [
    "ngram_candidates.csv",
    "candidate_phrase_review_table.csv",
    "curation_decisions.csv",
    "final_dish_list.csv",
    "task3_comparison_summary.csv",
]
missing = [f for f in expected_artifacts if not (ARTIFACT_DIR / f).exists()]
if missing:
    print("⚠  Missing artifacts:", missing)
else:
    print("✓ All required artifacts present.")
for f in expected_artifacts:
    p = ARTIFACT_DIR / f
    if p.exists():
        try:
            nrows = len(pd.read_csv(p))
            print(f"  OK: {f}: {nrows} rows")
        except Exception:
            print(f"  OK: {f} (unreadable)")
    else:
        print(f"  MISSING: {f}")